# Notebook 3 — Lightweight Ablation: BoW + ConvNeXt-Small + Conv TokenLearner

| Component | Choice |
|---|---|
| Vision Encoder | `ConvNeXt-Small (dims=(48,96,192))` |
| Text Encoder | `BoW 256-dim (fully trainable)` |
| Token Learner | `ConvTokenLearner — no attention (16 tokens)` |
| Goal | Fastest training, minimum VRAM, pure-CNN ablation |

> XAI outputs saved to `xai_outputs_{MODEL_NAME}/`

In [1]:
# ========================= BLOCK 1: IMPORTS =========================
import sys, os, gc, time, math, random, warnings, subprocess
import numpy as np
from collections import Counter, defaultdict

# ── Force TensorFlow to CPU BEFORE importing it ──
os.environ['CUDA_VISIBLE_DEVICES_TF'] = ''  # Some TF builds respect this
import tensorflow as tf
tf.config.set_visible_devices([], 'GPU')  # Prevent TF from touching GPU

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.backends.cudnn as cudnn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms
from einops import rearrange, reduce, repeat
from einops.layers.torch import Rearrange, Reduce
from transformers import (
    DistilBertModel, DistilBertTokenizer,
    BertModel, BertTokenizer,
    CLIPTokenizer, CLIPTextModel,
)
from tqdm.auto import tqdm
from PIL import Image as PILImage

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

import tensorflow_datasets as tfds

warnings.filterwarnings('ignore')

# H100 / GPU optimizations
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32       = True
cudnn.benchmark = True
torch.set_float32_matmul_precision('high')
torch.set_num_threads(8)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
if torch.cuda.is_available():
    print(f"  GPU:  {torch.cuda.get_device_name(0)}")
    print(f"  VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.0f} GB")
print("✅ Block 1 done")

2026-03-03 02:02:57.436327: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-03 02:02:57.476765: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-03-03 02:02:58.441234: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


Device: cuda
  GPU:  NVIDIA GeForce RTX 3060 Laptop GPU
  VRAM: 6 GB
✅ Block 1 done


In [2]:

# ========================= BLOCK 2: CONFIGURATION =========================
SEED=42; torch.manual_seed(SEED); np.random.seed(SEED); random.seed(SEED)
if DEVICE=="cuda": torch.cuda.manual_seed_all(SEED)

TARGET_SIZE=224; SEQ_LEN=6; SPLIT_RATIO=0.8; ACTION_BINS=256
TEXT_DIM=256; TRANSFORMER_DEPTH=4; TRANSFORMER_HEADS=4
DROPOUT=0.2; NUM_VISUAL_TOKENS=16
BATCH_SIZE=4; GRADIENT_ACCUM_STEPS=32; EPOCHS=80; LR=1e-4; WEIGHT_DECAY=0.1
LION_BETAS=(0.9,0.99); WARMUP_STEPS=300; GRAD_CLIP=1.0
LABEL_SMOOTHING=0.1; MIXUP_ALPHA=0.3; USE_CUTMIX=True; FOCAL_GAMMA=2.5; PATIENCE=10
NUM_WORKERS=2; PREFETCH_FACTOR=4; USE_BFLOAT16=True; AUTOCAST_DTYPE=torch.bfloat16
MODEL_NAME="rt1_lightweight_bow_convnext"; VIS_ENC_NAME="ConvNeXt-Small"; TEXT_ENC_NAME="BoW-256"; TL_NAME="ConvTL"
print("✅ Block 2 done")


✅ Block 2 done


In [3]:
# ========================= BLOCK 3: DOWNLOAD DATASET =========================
DATASET_NAME = "berkeley_cable_routing"
DATA_DIR     = os.path.join(os.getcwd(), "berkeley_cable_routing_data")

def check_existing():
    if not os.path.exists(DATA_DIR): return False
    for root, dirs, files in os.walk(DATA_DIR):
        if "dataset_info.json" in files:
            total = sum(os.path.getsize(os.path.join(root,f)) for f in files)/1e9
            print(f"✅ Already downloaded: {root} ({total:.2f} GB)"); return True
    return False

def download_gcs_tf():
    src = f"gs://gresearch/robotics/{DATASET_NAME}"
    print(f"📥 Downloading from {src}")
    def list_all(d):
        out=[]
        for item in tf.io.gfile.listdir(d):
            c=item.rstrip('/');
            if '$folder$' in c: continue
            full=f"{d}/{c}"
            if tf.io.gfile.isdir(full): out.extend(list_all(full))
            else: out.append(full)
        return out
    files=list_all(src); print(f"   Found {len(files)} files")
    for i,p in enumerate(files):
        rel=p[len(src)+1:]; local=os.path.join(DATA_DIR,rel)
        os.makedirs(os.path.dirname(local),exist_ok=True)
        if os.path.exists(local):
            try:
                if os.path.getsize(local)==tf.io.gfile.stat(p).length: continue
            except: pass
        tf.io.gfile.copy(p,local,overwrite=True)
        print(f"   [{i+1}/{len(files)}] {rel}")
    print("✅ Download complete")

if not check_existing(): download_gcs_tf()

BUILDER_DIR=None
for root,dirs,files in os.walk(DATA_DIR):
    if "dataset_info.json" in files: BUILDER_DIR=root; break
if BUILDER_DIR is None: raise FileNotFoundError(f"dataset_info.json not in {DATA_DIR}")

# Force builder creation on CPU
with tf.device('/CPU:0'):
    builder = tfds.builder_from_directory(builder_dir=BUILDER_DIR)
for s,si in builder.info.splits.items(): print(f"   {s}: {si.num_examples:,} episodes")
print("✅ Block 3 done")

✅ Already downloaded: /media/sparsh/CaptainSlow/Programming Stuff/Programming Stuff/Code _n_ Stuff/All Projects/All Projects/VLAs/my work/berkeley_cable_routing_data/0.1.0 (5.01 GB)
   train: 1,482 episodes
   test: 165 episodes
✅ Block 3 done


In [4]:
# ========================= BLOCK 4: EXPLORE STRUCTURE =========================
# Force TensorFlow to CPU-only to avoid VRAM consumption
import tensorflow as tf
tf.config.set_visible_devices([], 'GPU')  # Hide all GPUs from TF

ds_peek = builder.as_dataset(split="train[:1]")
ep=next(iter(ds_peek)); steps=list(ep["steps"]); step=steps[0]
obs,act=step["observation"],step["action"]

IMG_KEY=None
for k,v in obs.items():
    if hasattr(v,'shape') and len(v.shape)==3 and v.shape[-1]==3:
        IMG_KEY=k; IMG_SHAPE=tuple(v.shape)

ACT_KEYS=sorted(act.keys())
sample_action=np.concatenate([np.atleast_1d(act[k].numpy()).astype(np.float32) for k in ACT_KEYS])
ACTUAL_ACTION_DIM=len(sample_action); NUM_ACTIONS=ACTUAL_ACTION_DIM

INST_KEY=None
for k,v in obs.items():
    val=v.numpy() if hasattr(v,'numpy') else v
    if isinstance(val,bytes): INST_KEY=k; break
    if isinstance(val,np.ndarray) and val.dtype.kind in ('U','S','O'): INST_KEY=k; break

print(f"Image: '{IMG_KEY}' {IMG_SHAPE}")
print(f"Action: {ACT_KEYS}, dim={ACTUAL_ACTION_DIM}")
print(f"Instruction: '{INST_KEY}'")
del ds_peek,ep,steps; gc.collect()
print("✅ Block 4 done")

Image: 'wrist45_image' (128, 128, 3)
Action: ['rotation_delta', 'terminate_episode', 'world_vector'], dim=7
Instruction: 'natural_language_instruction'
✅ Block 4 done


2026-03-03 02:03:28.890677: I tensorflow/core/kernels/data/tf_record_dataset_op.cc:396] The default buffer size is 262144, which is overridden by the user specified `buffer_size` of 8388608
2026-03-03 02:03:28.928972: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


In [5]:
# ========================= BLOCK 5: UTILITIES =========================
class RT1Tokenizer:
    def __init__(self, vocab_size=256, range_min=-3.2, range_max=3.2):
        self.vocab_size=vocab_size; self.min=range_min; self.max=range_max
    def encode(self, af):
        af=torch.tensor(af,dtype=torch.float32) if not isinstance(af,torch.Tensor) else af.float()
        af=torch.clamp(af,self.min,self.max)
        return ((af-self.min)/(self.max-self.min)*(self.vocab_size-1)).long()
    def decode(self, t):
        return t.float()/(self.vocab_size-1)*(self.max-self.min)+self.min

class DropPath(nn.Module):
    def __init__(self, p=0.0): super().__init__(); self.p=p
    def forward(self, x):
        if self.p==0. or not self.training: return x
        keep=1-self.p; shape=(x.shape[0],)+(1,)*(x.ndim-1)
        mask=(torch.rand(shape,dtype=x.dtype,device=x.device)+keep).floor_()
        return x/keep*mask

print("✅ Block 5 done")


✅ Block 5 done


In [6]:
# ========================= BLOCK 6: MODEL COMPONENTS =========================

# ─── Squeeze-Excitation ───────────────────────────────────────────────────────
class ImprovedSqueezeExcitation(nn.Module):
    def __init__(self, dim, reduction_ratio=4):
        super().__init__()
        rd=max(1,dim//reduction_ratio)
        self.se=nn.Sequential(
            Reduce('b c h w -> b c','mean'),
            nn.Linear(dim,rd,bias=False),nn.BatchNorm1d(rd),nn.SiLU(),nn.Dropout(0.1),
            nn.Linear(rd,dim,bias=False),nn.Sigmoid(),Rearrange('b c -> b c 1 1'))
    def forward(self,x): return x*self.se(x)

class ImprovedMBConv(nn.Module):
    def __init__(self,dim_in,dim_out,*,downsample,expansion_rate=4,use_fused=False,drop_path_rate=0.1,se_ratio=0.25):
        super().__init__()
        self.use_residual=(dim_in==dim_out) and (not downsample)
        hidden=int(expansion_rate*dim_in); stride=2 if downsample else 1; se_rd=int(1/se_ratio)
        if use_fused:
            self.net=nn.Sequential(
                nn.Conv2d(dim_in,hidden,3,stride=stride,padding=1,bias=False),
                nn.BatchNorm2d(hidden),nn.SiLU(),
                ImprovedSqueezeExcitation(hidden,se_rd),
                nn.Conv2d(hidden,dim_out,1,bias=False),nn.BatchNorm2d(dim_out))
        else:
            self.net=nn.Sequential(
                nn.Conv2d(dim_in,hidden,1,bias=False),nn.BatchNorm2d(hidden),nn.SiLU(),
                nn.Conv2d(hidden,hidden,3,stride=stride,padding=1,groups=hidden,bias=False),
                nn.BatchNorm2d(hidden),nn.SiLU(),
                ImprovedSqueezeExcitation(hidden,se_rd),
                nn.Conv2d(hidden,dim_out,1,bias=False),nn.BatchNorm2d(dim_out))
        self.drop_path=DropPath(drop_path_rate) if (drop_path_rate>0 and self.use_residual) else nn.Identity()
    def forward(self,x): return x+self.drop_path(self.net(x)) if self.use_residual else self.net(x)

class ImprovedTemporalConv(nn.Module):
    def __init__(self,dim):
        super().__init__()
        self.temporal_conv=nn.Sequential(
            nn.Conv3d(dim,dim,(3,1,1),padding=(1,0,0),groups=dim),nn.BatchNorm3d(dim),nn.GELU())
        self.temporal_attn=nn.Sequential(
            nn.AdaptiveAvgPool3d(1),nn.Conv3d(dim*2,dim,1),nn.GELU(),nn.Conv3d(dim,dim*2,1),nn.Sigmoid())
        self.proj=nn.Conv3d(dim*2,dim,1)
    def forward(self,x):
        cat=torch.cat([x,self.temporal_conv(x)],dim=1)
        return x+self.proj(cat*self.temporal_attn(cat))

# ─── LayerNorm2d: channel-wise LayerNorm for NCHW tensors ────────────────────
# Standard nn.LayerNorm(d) on (B,C,H,W) normalizes over W (wrong!).
# LayerNorm2d permutes to (B,H,W,C), normalizes over C, then permutes back.
class LayerNorm2d(nn.Module):
    def __init__(self,dim,eps=1e-6):
        super().__init__()
        self.norm=nn.LayerNorm(dim,eps=eps)
    def forward(self,x):
        return self.norm(x.permute(0,2,3,1)).permute(0,3,1,2)

# ─── ConvNeXt ─────────────────────────────────────────────────────────────────
class ConvNeXtBlock(nn.Module):
    def __init__(self,dim,drop_path_rate=0.0,expand=4):
        super().__init__()
        self.dw=nn.Conv2d(dim,dim,7,padding=3,groups=dim)
        self.norm=nn.LayerNorm(dim,eps=1e-6)
        self.pw1=nn.Linear(dim,dim*expand); self.pw2=nn.Linear(dim*expand,dim); self.act=nn.GELU()
        self.dp=DropPath(drop_path_rate) if drop_path_rate>0 else nn.Identity()
        self.gamma=nn.Parameter(1e-6*torch.ones(dim))
    def forward(self,x):
        r=x; x=self.dw(x); x=x.permute(0,2,3,1); x=self.norm(x)
        x=self.pw2(self.act(self.pw1(x)))*self.gamma; x=x.permute(0,3,1,2)
        return r+self.dp(x)

class ConvNeXtEncoder(nn.Module):
    CONFIGS={'nano' :dict(dims=(32, 64),       blocks=(2,2,0)),
             'small':dict(dims=(48, 96, 192),   blocks=(2,2,3)),
             'base' :dict(dims=(64, 128, 256),  blocks=(3,3,6))}
    def __init__(self,size='small',drop_path_rate=0.15):
        super().__init__()
        cfg=self.CONFIGS[size]; dims,blocks=cfg['dims'],cfg['blocks']
        # Stem: patchify 4×4, then LayerNorm2d over dims[0] channels
        self.stem=nn.Sequential(
            nn.Conv2d(3,dims[0],kernel_size=4,stride=4,bias=False),
            LayerNorm2d(dims[0]))
        total=sum(blocks); blk=0; stages=[]
        for si,(d,n) in enumerate(zip(dims,blocks)):
            stage=[]
            # Downsampler between stages:
            #   - LayerNorm2d over PREVIOUS stage channels (dims[si-1])
            #   - Conv2d projects dims[si-1] → dims[si], stride=2
            if si>0:
                prev_d=dims[si-1]
                stage.append(nn.Sequential(
                    LayerNorm2d(prev_d),
                    nn.Conv2d(prev_d,d,kernel_size=2,stride=2)))
            for _ in range(n):
                dp=drop_path_rate*blk/max(total-1,1)
                stage.append(ConvNeXtBlock(d,dp)); blk+=1
            stages.append(nn.Sequential(*stage))
        self.stages=nn.ModuleList(stages)
        self.embed_dim=dims[-1]
        self.final_norm=LayerNorm2d(dims[-1])
    def forward(self,x):
        x=self.stem(x)                     # (B, dims[0], H/4, W/4)
        for stage in self.stages: x=stage(x)
        return self.final_norm(x)          # (B, embed_dim, H', W')

# ─── EfficientLite ────────────────────────────────────────────────────────────
class DSConvBlock(nn.Module):
    def __init__(self,dim_in,dim_out,stride=1,drop_path=0.0):
        super().__init__()
        self.use_res=(dim_in==dim_out) and (stride==1)
        self.dw=nn.Sequential(nn.Conv2d(dim_in,dim_in,3,stride=stride,padding=1,groups=dim_in,bias=False),
                               nn.BatchNorm2d(dim_in),nn.SiLU())
        self.pw=nn.Sequential(nn.Conv2d(dim_in,dim_out,1,bias=False),nn.BatchNorm2d(dim_out))
        self.dp=DropPath(drop_path) if (drop_path>0 and self.use_res) else nn.Identity()
    def forward(self,x):
        out=self.pw(self.dw(x)); return (x+self.dp(out)) if self.use_res else out

class EfficientLiteEncoder(nn.Module):
    CONFIGS={'xs':dict(dims=(16,32,48),blocks=(2,2,2)),
             's':dict(dims=(24,48,72),blocks=(3,3,3)),
             'm':dict(dims=(32,64,96),blocks=(3,4,4))}
    def __init__(self,size='s',drop_path_rate=0.1):
        super().__init__()
        cfg=self.CONFIGS[size]; dims,blocks=cfg['dims'],cfg['blocks']
        self.stem=nn.Sequential(nn.Conv2d(3,dims[0],3,stride=2,padding=1,bias=False),
                                 nn.BatchNorm2d(dims[0]),nn.SiLU())
        total=sum(blocks); blk=0; stages=[]
        for si,(d_in,n) in enumerate(zip(dims,blocks)):
            d_out=dims[si+1] if si+1<len(dims) else d_in
            stage=[]; stride=2 if si>0 else 1
            dp=drop_path_rate*blk/max(total-1,1)
            stage.append(DSConvBlock(d_in,d_out if si>0 else d_in,stride=stride,drop_path=dp)); blk+=1
            d=d_out if si>0 else d_in
            for _ in range(n-1):
                dp=drop_path_rate*blk/max(total-1,1)
                stage.append(DSConvBlock(d,d,drop_path=dp)); blk+=1
            stages.append(nn.Sequential(*stage))
        self.stages=nn.ModuleList(stages); self.embed_dim=dims[-1]
    def forward(self,x):
        x=self.stem(x)
        for stage in self.stages: x=stage(x)
        return x

# ─── Token Learners ───────────────────────────────────────────────────────────
class AdvancedTokenLearner(nn.Module):
    def __init__(self,dim,num_tokens=32,num_heads=4):
        super().__init__()
        self.token_queries=nn.Parameter(torch.randn(1,num_tokens,dim))
        self.pos_embed=nn.Parameter(torch.randn(1,dim,7,7))
        self.cross_attn=nn.MultiheadAttention(dim,num_heads,dropout=0.1,batch_first=True)
        self.ffn=nn.Sequential(nn.LayerNorm(dim),nn.Linear(dim,dim*4),nn.GELU(),
                                nn.Dropout(0.1),nn.Linear(dim*4,dim),nn.Dropout(0.1))
        self.norm1=nn.LayerNorm(dim); self.norm2=nn.LayerNorm(dim)
    def forward(self,x):
        b,c,h,w=x.shape
        pos=F.interpolate(self.pos_embed,(h,w),mode='bilinear',align_corners=False) if (h,w)!=(7,7) else self.pos_embed
        spatial=rearrange(x+pos,'b c h w -> b (h w) c')
        q=self.token_queries.expand(b,-1,-1)
        out,_=self.cross_attn(query=q,key=spatial,value=spatial)
        tokens=self.norm1(q+out); return self.norm2(tokens+self.ffn(tokens))

class ConvTokenLearner(nn.Module):
    def __init__(self,dim,num_tokens=32,num_heads=4):
        super().__init__()
        self.num_tokens=num_tokens
        self.attn_maps=nn.Sequential(
            nn.Conv2d(dim,num_tokens,1,bias=False),nn.BatchNorm2d(num_tokens),nn.GELU(),
            nn.Conv2d(num_tokens,num_tokens,1,bias=False),nn.Sigmoid())
        self.norm=nn.LayerNorm(dim)
    def forward(self,x):
        B,C,H,W=x.shape
        attn=self.attn_maps(x); attn=attn/(attn.sum(dim=(2,3),keepdim=True)+1e-8)
        x_flat=x.view(B,C,H*W); attn_flat=attn.view(B,self.num_tokens,H*W)
        tokens=torch.bmm(attn_flat,x_flat.permute(0,2,1))
        return self.norm(tokens)

class HierarchicalTokenLearner(nn.Module):
    def __init__(self,dim,num_tokens=32,num_heads=4):
        super().__init__()
        n_fine=num_tokens*3//4; n_coarse=num_tokens-n_fine
        self.fine=ConvTokenLearner(dim,n_fine)
        self.pool=nn.AvgPool2d(2,stride=2)
        self.coarse=ConvTokenLearner(dim,n_coarse)
        self.num_tokens=num_tokens; self.norm=nn.LayerNorm(dim)
    def forward(self,x):
        return self.norm(torch.cat([self.fine(x),self.coarse(self.pool(x))],dim=1))

class SelfAttentionTokenLearner(nn.Module):
    def __init__(self,dim,num_tokens=32,num_heads=4):
        super().__init__()
        self.queries=nn.Parameter(torch.randn(1,num_tokens,dim)*0.02)
        self.self_attn=nn.MultiheadAttention(dim,num_heads,dropout=0.1,batch_first=True)
        self.norm1=nn.LayerNorm(dim)
        self.cross_attn=nn.MultiheadAttention(dim,num_heads,dropout=0.1,batch_first=True)
        self.norm2=nn.LayerNorm(dim)
        self.pos=nn.Parameter(torch.randn(1,1,dim)*0.02)
    def forward(self,x):
        B,C,H,W=x.shape; spatial=x.view(B,C,H*W).permute(0,2,1)+self.pos
        sa,_=self.self_attn(spatial,spatial,spatial); spatial=self.norm1(spatial+sa)
        q=self.queries.expand(B,-1,-1); ca,_=self.cross_attn(q,spatial,spatial)
        return self.norm2(q+ca)

# ─── Text Encoders ────────────────────────────────────────────────────────────
class DistilBERTEncoder:
    def __init__(self,device='cuda',freeze=True):
        self.device=device; self.frozen=freeze
        self.tokenizer=DistilBertTokenizer.from_pretrained('distilbert-base-uncased')
        self.model=DistilBertModel.from_pretrained('distilbert-base-uncased').to(device)
        self.embed_dim=768
        if freeze: [p.__setattr__('requires_grad',False) for p in self.model.parameters()]; self.model.eval()
        print(f"   ✅ DistilBERT (768-dim, frozen={freeze})")
    def encode(self,instructions):
        if isinstance(instructions,str): instructions=[instructions]
        enc=self.tokenizer(instructions,padding=True,truncation=True,max_length=77,return_tensors='pt').to(self.device)
        ctx=torch.no_grad() if self.frozen else torch.enable_grad()
        with ctx: out=self.model(**enc)
        return out.last_hidden_state[:,0,:]

class BERTEncoder:
    def __init__(self,device='cuda',freeze=True,unfreeze_top_k=0):
        self.device=device; self.frozen=freeze
        self.tokenizer=BertTokenizer.from_pretrained('bert-base-uncased')
        self.model=BertModel.from_pretrained('bert-base-uncased').to(device)
        self.embed_dim=768
        if freeze:
            for p in self.model.parameters(): p.requires_grad=False
            if unfreeze_top_k>0:
                n=len(self.model.encoder.layer)
                for layer in self.model.encoder.layer[n-unfreeze_top_k:]:
                    for p in layer.parameters(): p.requires_grad=True
                self.frozen=f'partial(top-{unfreeze_top_k})'
            else: self.model.eval()
        print(f"   ✅ BERT (768-dim, frozen={self.frozen})")
    def encode(self,instructions):
        if isinstance(instructions,str): instructions=[instructions]
        enc=self.tokenizer(instructions,padding=True,truncation=True,max_length=77,return_tensors='pt').to(self.device)
        ctx=torch.no_grad() if self.frozen is True else torch.enable_grad()
        with ctx: out=self.model(**enc)
        return out.last_hidden_state[:,0,:]

class CLIPTextEncoder:
    def __init__(self,device='cuda',freeze=True):
        self.device=device; self.frozen=freeze
        self.tokenizer=CLIPTokenizer.from_pretrained('openai/clip-vit-base-patch32')
        self.model=CLIPTextModel.from_pretrained('openai/clip-vit-base-patch32').to(device)
        self.embed_dim=512
        if freeze: [p.__setattr__('requires_grad',False) for p in self.model.parameters()]; self.model.eval()
        print(f"   ✅ CLIP Text (512-dim, frozen={freeze})")
    def encode(self,instructions):
        if isinstance(instructions,str): instructions=[instructions]
        enc=self.tokenizer(instructions,padding=True,truncation=True,max_length=77,return_tensors='pt').to(self.device)
        ctx=torch.no_grad() if self.frozen else torch.enable_grad()
        with ctx: out=self.model(**enc)
        return out.pooler_output

class BoWTextEncoder(nn.Module):
    def __init__(self,out_dim=256,vocab=8192,device='cuda',dropout=0.1):
        super().__init__()
        self.device=device; self.frozen=False; self.embed_dim=out_dim
        self._tok=DistilBertTokenizer.from_pretrained('distilbert-base-uncased')
        self.embed=nn.Embedding(vocab,128,padding_idx=0)
        self.mlp=nn.Sequential(nn.Linear(128,out_dim),nn.GELU(),nn.Dropout(dropout),
                                nn.Linear(out_dim,out_dim),nn.LayerNorm(out_dim))
        nn.init.normal_(self.embed.weight,std=0.02); self.to(device)
        print(f"   ✅ BoW ({out_dim}-dim, fully trainable)")
    def encode(self,instructions):
        if isinstance(instructions,str): instructions=[instructions]
        enc=self._tok(instructions,padding=True,truncation=True,max_length=77,return_tensors='pt').to(self.device)
        ids=enc['input_ids'].clamp(max=8191); pad=(ids==0)
        emb=self.embed(ids); mask=(~pad).unsqueeze(-1).float()
        pooled=(emb*mask).sum(1)/mask.sum(1).clamp(min=1)
        return self.mlp(pooled)

print("✅ Block 6 done — all model components defined")

✅ Block 6 done — all model components defined


In [7]:
# ========================= BLOCK 7: LIGHTWEIGHT MODEL =========================
vit=ConvNeXtEncoder(size='small', drop_path_rate=0.15)
print(f"   ConvNeXt-Small embed_dim={vit.embed_dim}")

class LightweightRT1(nn.Module):
    def __init__(self,vit,text_dim=256,num_actions=7,action_bins=256,depth=4,heads=4,dropout=0.2,num_visual_tokens=16):
        super().__init__()
        self.vit=vit; self.num_actions=num_actions; self.action_bins=action_bins; self.embed_dim=vit.embed_dim
        self.temporal_model=ImprovedTemporalConv(self.embed_dim)
        self.token_learner=ConvTokenLearner(self.embed_dim,num_visual_tokens)
        self.text_proj=nn.Sequential(nn.Linear(text_dim,self.embed_dim,bias=False),nn.LayerNorm(self.embed_dim),nn.Dropout(dropout))
        enc_layer=nn.TransformerEncoderLayer(d_model=self.embed_dim,nhead=heads,dim_feedforward=self.embed_dim*4,
                                              dropout=dropout,batch_first=True,norm_first=True,activation='gelu')
        self.transformer=nn.TransformerEncoder(enc_layer,depth)
        self.to_logits=nn.Sequential(nn.LayerNorm(self.embed_dim),nn.Linear(self.embed_dim,self.embed_dim//2),
                                      nn.GELU(),nn.Dropout(dropout),nn.Linear(self.embed_dim//2,num_actions*action_bins))
    def forward(self,video,text_embeddings):
        b,c,f,h,w=video.shape; flat=rearrange(video,'b c f h w -> (b f) c h w')
        vit_out=self.vit(flat); vit_out=rearrange(vit_out,'(b f) d h w -> b d f h w',b=b,f=f)
        vit_out=self.temporal_model(vit_out); vit_out=rearrange(vit_out,'b d f h w -> (b f) d h w')
        tokens=self.token_learner(vit_out); tokens=rearrange(tokens,'(b f) n d -> b (f n) d',b=b,f=f)
        text_tok=self.text_proj(text_embeddings).unsqueeze(1)
        attended=self.transformer(torch.cat([text_tok,tokens],dim=1))
        logits=self.to_logits(attended[:,0,:])
        return rearrange(logits,'b (a bin) -> b a bin',a=self.num_actions)

model=LightweightRT1(vit=vit,text_dim=TEXT_DIM,num_actions=NUM_ACTIONS,action_bins=ACTION_BINS,
                     depth=TRANSFORMER_DEPTH,heads=TRANSFORMER_HEADS,dropout=DROPOUT,
                     num_visual_tokens=NUM_VISUAL_TOKENS).to(DEVICE)
total_p=sum(p.numel() for p in model.parameters())
print(f"🤖 Lightweight-BoW-ConvNeXt: {total_p:,} params  embed_dim={model.embed_dim}")
with torch.no_grad():
    dv=torch.randn(2,3,SEQ_LEN,TARGET_SIZE,TARGET_SIZE).to(DEVICE); dt=torch.randn(2,TEXT_DIM).to(DEVICE)
    print(f"   Sanity: {model(dv,dt).shape}"); del dv,dt
torch.cuda.empty_cache(); print("✅ Block 7 done")


   ConvNeXt-Small embed_dim=192
🤖 Lightweight-BoW-ConvNeXt: 3,463,264 params  embed_dim=192
   Sanity: torch.Size([2, 7, 256])
✅ Block 7 done


In [8]:
# ========================= BLOCK 8: TRAINING UTILITIES =========================
class ImprovedFocalLoss(nn.Module):
    def __init__(self,gamma=2.5,label_smoothing=0.1,temperature=1.0):
        super().__init__()
        self.gamma=gamma; self.label_smoothing=label_smoothing; self.temperature=temperature
    def forward(self,logits,targets):
        logits=logits/self.temperature
        ce=F.cross_entropy(logits,targets,reduction='none',label_smoothing=self.label_smoothing)
        p_t=F.softmax(logits,dim=-1).gather(1,targets.unsqueeze(1)).squeeze(1)
        return ((1-p_t)**self.gamma*ce).mean()

class ImprovedMixup(nn.Module):
    def __init__(self,alpha=0.3,use_cutmix=True,cutmix_prob=0.5):
        super().__init__()
        self.alpha=alpha; self.use_cutmix=use_cutmix; self.cutmix_prob=cutmix_prob
    def forward(self,x,y):
        lam=np.random.beta(self.alpha,self.alpha) if self.alpha>0 else 1.0
        idx=torch.randperm(x.size(0),device=x.device)
        if self.use_cutmix and np.random.rand()<self.cutmix_prob:
            W,H=x.size(-1),x.size(-2); cut=int(W*np.sqrt(1-lam))
            cx,cy=np.random.randint(W),np.random.randint(H)
            x1,y1=np.clip(cx-cut//2,0,W),np.clip(cy-cut//2,0,H)
            x2,y2=np.clip(cx+cut//2,0,W),np.clip(cy+cut//2,0,H)
            x[:,:,:,y1:y2,x1:x2]=x[idx,:,:,y1:y2,x1:x2]
            lam=1-(x2-x1)*(y2-y1)/(W*H)
        else: x=lam*x+(1-lam)*x[idx]
        return x,y,y[idx],lam

class Lion(torch.optim.Optimizer):
    def __init__(self,params,lr=1e-4,betas=(0.9,0.99),weight_decay=0.0):
        super().__init__(params,dict(lr=lr,betas=betas,weight_decay=weight_decay))
    @torch.no_grad()
    def step(self,closure=None):
        for group in self.param_groups:
            for p in group['params']:
                if p.grad is None: continue
                if group['weight_decay']>0: p.data.mul_(1-group['lr']*group['weight_decay'])
                state=self.state[p]
                if len(state)==0: state['exp_avg']=torch.zeros_like(p)
                exp_avg=state['exp_avg']; b1,b2=group['betas']
                update=exp_avg*b1+p.grad*(1-b1)
                p.add_(torch.sign(update),alpha=-group['lr'])
                exp_avg.mul_(b2).add_(p.grad,alpha=1-b2)

class ImprovedScheduler:
    def __init__(self,optimizer,warmup_steps=500,max_steps=10000,min_lr_ratio=0.1):
        self.optimizer=optimizer; self.warmup=warmup_steps; self.max_steps=max_steps
        self.min_ratio=min_lr_ratio; self.base_lrs=[g['lr'] for g in optimizer.param_groups]; self.step_count=0
    def step(self):
        self.step_count+=1
        for pg,blr in zip(self.optimizer.param_groups,self.base_lrs):
            if self.step_count<self.warmup: pg['lr']=blr*self.step_count/self.warmup
            else:
                prog=(self.step_count-self.warmup)/max(1,self.max_steps-self.warmup)
                pg['lr']=self.min_ratio*blr+(blr-self.min_ratio*blr)*0.5*(1+math.cos(math.pi*prog))
    def get_last_lr(self): return [g['lr'] for g in self.optimizer.param_groups]

class ModelEMA:
    def __init__(self,model,decay=0.9999):
        self.decay=decay; self.model=model
        self.shadow={n:p.data.clone() for n,p in model.named_parameters() if p.requires_grad}; self.backup={}
    def update(self,model):
        for n,p in model.named_parameters():
            if p.requires_grad: self.shadow[n]=self.decay*self.shadow[n]+(1-self.decay)*p.data
    def apply_shadow(self):
        for n,p in self.model.named_parameters():
            if p.requires_grad: self.backup[n]=p.data; p.data=self.shadow[n]
    def restore(self):
        for n,p in self.model.named_parameters():
            if p.requires_grad: p.data=self.backup[n]
        self.backup={}

class PerTaskMetrics:
    def __init__(self): self.task_losses=defaultdict(list)
    def reset(self): self.task_losses=defaultdict(list)
    def update(self,losses,task_ids):
        for l,t in zip(losses,task_ids): self.task_losses[t.item()].append(l.item())
    def get_average(self): return {t:np.mean(ls) for t,ls in self.task_losses.items()}

print("✅ Block 8 done")


✅ Block 8 done


In [9]:
# ========================= BLOCK 9: DATASET =========================
class CableRoutingDataset(Dataset):
    def __init__(self,split,mode="train",seq_len=SEQ_LEN,target_size=TARGET_SIZE,max_episodes=None):
        self.seq_len=seq_len; self.target_size=target_size; self.mode=mode; self.tokenizer=RT1Tokenizer()
        self.cache_dir=os.path.join(os.getcwd(),f"cable_cache_{mode}")
        self.img_path=os.path.join(self.cache_dir,"images.dat")
        self.act_path=os.path.join(self.cache_dir,"actions.npy")
        self.inst_path=os.path.join(self.cache_dir,"instructions.npy")
        self.meta_path=os.path.join(self.cache_dir,"metadata.npy")
        if os.path.exists(self.meta_path): self._load_cache()
        else: self._build_cache(split,max_episodes)
        self.indices=[]
        for ep_s,ep_e in self.episode_bounds:
            if ep_e-ep_s>=seq_len:
                for i in range(ep_s+seq_len,ep_e+1): self.indices.append(i)
        self._build_task_index(); self._setup_augmentation()
        size_gb=os.path.getsize(self.img_path)/1e9 if os.path.exists(self.img_path) else 0
        print(f"   ✅ {mode.upper()}: {len(self.episode_bounds)} ep, {len(self.indices):,} samples, "
              f"{self.num_tasks} tasks, {size_gb:.2f} GB")

    def _build_cache(self,split,max_episodes):
        os.makedirs(self.cache_dir,exist_ok=True)
        print(f"\n📂 Building cache: {self.mode} → {self.cache_dir}")
        ds=builder.as_dataset(split=split)
        total_frames,ep_count=0,0
        for ep in tqdm(ds,desc="   Counting"):
            if max_episodes and ep_count>=max_episodes: break
            total_frames+=sum(1 for _ in ep["steps"]); ep_count+=1
        print(f"   {ep_count} episodes, {total_frames} frames")
        ds2=builder.as_dataset(split=split)
        img_mmap=np.memmap(self.img_path,dtype=np.uint8,mode='w+',
                           shape=(total_frames,self.target_size,self.target_size,3))
        actions_list,instructions_list,episode_bounds=[],[],[]
        idx,ep_num=0,0
        for ep in tqdm(ds2,desc="   Writing",total=ep_count):
            if max_episodes and ep_num>=max_episodes: break
            ep_start=idx
            for step in ep["steps"]:
                img=step["observation"][IMG_KEY].numpy()
                if img.shape[0]!=self.target_size or img.shape[1]!=self.target_size:
                    img=np.array(PILImage.fromarray(img).resize((self.target_size,self.target_size),PILImage.BILINEAR))
                img_mmap[idx]=img
                act_vec=[np.atleast_1d(step["action"][k].numpy()).astype(np.float32) for k in ACT_KEYS]
                actions_list.append(np.concatenate(act_vec))
                inst=""
                if INST_KEY:
                    raw=step["observation"][INST_KEY].numpy()
                    inst=raw.decode() if isinstance(raw,bytes) else str(raw)
                instructions_list.append(inst or "route the cable")
                idx+=1
            episode_bounds.append((ep_start,idx)); ep_num+=1
        img_mmap.flush(); del img_mmap
        np.save(self.act_path,np.stack(actions_list))
        np.save(self.inst_path,np.array(instructions_list,dtype=object))
        np.save(self.meta_path,{'episode_bounds':episode_bounds,'n_frames':idx,'target_size':self.target_size})
        del actions_list,instructions_list; gc.collect(); self._load_cache()

    def _load_cache(self):
        meta=np.load(self.meta_path,allow_pickle=True).item()
        self.episode_bounds=meta['episode_bounds']; self.n_frames=meta['n_frames']
        self.all_images=np.memmap(self.img_path,dtype=np.uint8,mode='r',
                                  shape=(self.n_frames,self.target_size,self.target_size,3))
        self.all_actions=np.load(self.act_path)
        self.instructions=np.load(self.inst_path,allow_pickle=True).tolist()

    def _build_task_index(self):
        unique=sorted(set(self.instructions[i-1] for i in self.indices))
        self.instruction_to_id={inst:i for i,inst in enumerate(unique)}
        self.id_to_instruction={i:inst for inst,i in self.instruction_to_id.items()}
        self.idx_to_task={}; self.task_to_indices=defaultdict(list); self.task_counts=defaultdict(int)
        for idx in self.indices:
            inst=self.instructions[idx-1]; tid=self.instruction_to_id[inst]
            self.idx_to_task[idx]=tid; self.task_to_indices[tid].append(idx); self.task_counts[tid]+=1
        self.num_tasks=len(self.task_counts)

    def get_balanced_sampler(self):
        weights=np.array([1.0/self.task_counts[self.idx_to_task[i]] for i in self.indices])
        weights=weights/weights.sum()*len(weights)
        return WeightedRandomSampler(torch.DoubleTensor(weights),len(self.indices),replacement=True)

    def _setup_augmentation(self):
        if self.mode=="train":
            self.strong_aug=transforms.Compose([transforms.ToPILImage(),
                transforms.RandomApply([transforms.ColorJitter(0.4,0.4,0.4,0.2)],p=0.8),
                transforms.RandomResizedCrop(self.target_size,scale=(0.7,1.0)),
                transforms.RandomApply([transforms.GaussianBlur(5,(0.1,2.5))],p=0.5),
                transforms.ToTensor(),transforms.RandomErasing(p=0.3)])
            self.weak_aug=transforms.Compose([transforms.ToPILImage(),
                transforms.RandomResizedCrop(self.target_size,scale=(0.85,1.0)),transforms.ToTensor()])
        else:
            self.transform=transforms.Compose([transforms.ToPILImage(),
                transforms.Resize((self.target_size,self.target_size)),transforms.ToTensor()])

    def __len__(self): return len(self.indices)

    def __getitem__(self,idx):
        end=self.indices[idx]; start=end-self.seq_len
        instruction=self.instructions[end-1]; task_id=self.instruction_to_id[instruction]
        aug=self.strong_aug if (self.mode=="train" and self.task_counts[task_id]<500) else             (self.weak_aug if self.mode=="train" else self.transform)
        frames=[]
        for i in range(start,end):
            img=np.array(self.all_images[i])
            if img.dtype!=np.uint8:
                img=(img*255).astype(np.uint8) if img.max()<=1.0 else img.astype(np.uint8)
            frames.append(aug(img))
        video=torch.stack(frames); video=rearrange(video,'f c h w -> c f h w')
        action_tokens=self.tokenizer.encode(self.all_actions[end-1].astype(np.float32))
        return video,instruction,action_tokens,task_id

print("✅ Block 9 done")


✅ Block 9 done


In [10]:
# ========================= BLOCK 10: DATALOADERS =========================
print("\n" + "="*60)
print("INITIALIZING TEXT ENCODER + DATASETS")
print("="*60)

SKIP_TRAINING = True  # ← Set to True when you already have a checkpoint

text_encoder = BoWTextEncoder(out_dim=TEXT_DIM, device=DEVICE)

total_ep=builder.info.splits["train"].num_examples
train_ep=int(total_ep*SPLIT_RATIO)
print(f"\n📊 Split: {train_ep} train / {total_ep-train_ep} val")

with tf.device('/CPU:0'):
    dataset_train=CableRoutingDataset(split=f"train[:{train_ep}]",mode="train")
    dataset_val  =CableRoutingDataset(split=f"train[{train_ep}:]",mode="val")

train_sampler=dataset_train.get_balanced_sampler()

loader_train=DataLoader(dataset_train,batch_size=BATCH_SIZE,sampler=train_sampler,
    num_workers=NUM_WORKERS,pin_memory=True,drop_last=True,
    persistent_workers=NUM_WORKERS>0,prefetch_factor=PREFETCH_FACTOR if NUM_WORKERS>0 else None)
loader_val=DataLoader(dataset_val,batch_size=BATCH_SIZE,shuffle=False,
    num_workers=NUM_WORKERS,pin_memory=True,
    persistent_workers=NUM_WORKERS>0,prefetch_factor=PREFETCH_FACTOR if NUM_WORKERS>0 else None)

v,inst,tok,tid=dataset_train[0]
print(f"\n🧪 Sanity: video={v.shape}, tokens={tok.shape}")
print(f"✅ Block 10 done — {len(loader_train)} train / {len(loader_val)} val batches")


INITIALIZING TEXT ENCODER + DATASETS
   ✅ BoW (256-dim, fully trainable)

📊 Split: 1185 train / 297 val
   ✅ TRAIN: 1185 ep, 24,463 samples, 1 tasks, 4.56 GB
   ✅ VAL: 297 ep, 6,480 samples, 1 tasks, 1.19 GB

🧪 Sanity: video=torch.Size([3, 6, 224, 224]), tokens=torch.Size([7])
✅ Block 10 done — 6115 train / 1620 val batches


In [11]:
# ========================= BLOCK 11: TRAINING SETUP =========================
optimizer=Lion(model.parameters(),lr=LR,betas=LION_BETAS,weight_decay=WEIGHT_DECAY)
total_steps=len(loader_train)*EPOCHS//GRADIENT_ACCUM_STEPS
scheduler=ImprovedScheduler(optimizer,warmup_steps=WARMUP_STEPS,max_steps=total_steps)
criterion=ImprovedFocalLoss(gamma=FOCAL_GAMMA,label_smoothing=LABEL_SMOOTHING)
mixup=ImprovedMixup(alpha=MIXUP_ALPHA,use_cutmix=USE_CUTMIX)
ema=ModelEMA(model,decay=0.9999)
scaler=torch.amp.GradScaler('cuda',enabled=USE_BFLOAT16)
train_metrics=PerTaskMetrics(); val_metrics=PerTaskMetrics()
print(f"✅ Block 11 done — {total_steps} total steps")
print(f"   Lion LR={LR}, Focal γ={FOCAL_GAMMA}, EMA, Mixup+CutMix")


✅ Block 11 done — 15287 total steps
   Lion LR=0.0001, Focal γ=2.5, EMA, Mixup+CutMix


In [12]:
# ========================= BLOCK 12: TRAINING LOOP =========================
best_val_loss=float('inf'); patience_counter=0
history={'train_loss':[],'val_loss':[],'lr':[],'throughput':[]}

print("\n" + "="*60)
print(f"🚀 TRAINING: {MODEL_NAME}")
print("="*60 + "\n")

for epoch in range(EPOCHS):
    model.train(); train_loss=0.0; train_metrics.reset()
    optimizer.zero_grad(); epoch_t0=time.time(); samples=0

    loop=tqdm(loader_train,desc=f"E{epoch+1}/{EPOCHS} [TRAIN]")
    for batch_idx,(video,instructions,target,task_ids) in enumerate(loop):
        video=video.to(DEVICE,non_blocking=True)
        target=target.to(DEVICE,non_blocking=True)
        task_ids=task_ids.to(DEVICE,non_blocking=True)

        with torch.no_grad():
            text_emb=text_encoder.encode(list(instructions))

        video,target_a,target_b,lam=mixup(video,target)

        with torch.amp.autocast('cuda',dtype=AUTOCAST_DTYPE,enabled=USE_BFLOAT16):
            logits=model(video,text_emb)
            logits_flat=rearrange(logits,'b a bin -> (b a) bin')
            ta_flat=rearrange(target_a,'b a -> (b a)')
            tb_flat=rearrange(target_b,'b a -> (b a)')
            loss=(lam*criterion(logits_flat,ta_flat)+(1-lam)*criterion(logits_flat,tb_flat))/GRADIENT_ACCUM_STEPS

        scaler.scale(loss).backward()

        if (batch_idx+1)%GRADIENT_ACCUM_STEPS==0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(),GRAD_CLIP)
            scaler.step(optimizer); scaler.update()
            scheduler.step(); optimizer.zero_grad(); ema.update(model)

        train_loss+=loss.item()*GRADIENT_ACCUM_STEPS; samples+=video.size(0)
        loop.set_postfix(loss=f"{loss.item()*GRADIENT_ACCUM_STEPS:.3f}",
                         lr=f"{scheduler.get_last_lr()[0]:.2e}")

    # ── Validation ──
    model.eval(); val_loss=0.0; nb_val=0
    with torch.no_grad():
        for video,instructions,target,task_ids in loader_val:
            video=video.to(DEVICE); target=target.to(DEVICE)
            text_emb=text_encoder.encode(list(instructions))
            with torch.amp.autocast('cuda',dtype=AUTOCAST_DTYPE):
                logits=model(video,text_emb)
            val_loss+=criterion(rearrange(logits,'b a bin -> (b a) bin'),
                                rearrange(target,'b a -> (b a)')).item()
            nb_val+=1

    tl=train_loss/len(loader_train); vl=val_loss/max(nb_val,1)
    elapsed=time.time()-epoch_t0; thr=samples/elapsed
    history['train_loss'].append(tl); history['val_loss'].append(vl)
    history['lr'].append(scheduler.get_last_lr()[0]); history['throughput'].append(thr)
    print(f"  E{epoch+1:3d}/{EPOCHS} | train={tl:.4f} val={vl:.4f} | "
          f"{thr:.1f} samp/s | {elapsed:.0f}s")

    if vl<best_val_loss:
        best_val_loss=vl; patience_counter=0
        torch.save({'model':model.state_dict(),'epoch':epoch+1,'val_loss':best_val_loss},
                   f'{MODEL_NAME}_best.pt')
        print(f"    💾 Saved (val={best_val_loss:.4f})")
    else:
        patience_counter+=1
        if patience_counter>=PATIENCE: print(f"  ⏹ Early stop E{epoch+1}"); break

print(f"\n✅ Training done. Best val loss: {best_val_loss:.4f}")



🚀 TRAINING: rt1_lightweight_bow_convnext



E1/80 [TRAIN]:   0%|          | 0/6115 [00:00<?, ?it/s]

  E  1/80 | train=4.9562 val=4.1456 | 112.7 samp/s | 217s
    💾 Saved (val=4.1456)


E2/80 [TRAIN]:   0%|          | 0/6115 [00:00<?, ?it/s]

  E  2/80 | train=2.8980 val=1.9872 | 107.8 samp/s | 227s
    💾 Saved (val=1.9872)


...

  E 19/80 | train=1.8983 val=1.8960 | 107.9 samp/s | 227s


E20/80 [TRAIN]:   0%|          | 0/6115 [00:00<?, ?it/s]

  E 20/80 | train=1.8875 val=1.8960 | 108.0 samp/s | 226s


E21/80 [TRAIN]:   0%|          | 0/6115 [00:00<?, ?it/s]

  E 21/80 | train=1.8935 val=1.8954 | 108.1 samp/s | 226s
  ⏹ Early stop E21

✅ Training done. Best val loss: 1.8941


In [13]:
# ========================= BLOCK 13: TRAINING PLOTS =========================
fig,axes=plt.subplots(1,3,figsize=(18,5))
ne=len(history['train_loss']); ep_range=range(1,ne+1)
axes[0].plot(ep_range,history['train_loss'],label='Train',lw=2,marker='o',ms=3)
axes[0].plot(ep_range,history['val_loss'],label='Val',lw=2,marker='s',ms=3)
axes[0].set_title(f'Loss — {MODEL_NAME}',fontsize=14,fontweight='bold')
axes[0].set_xlabel('Epoch'); axes[0].legend(); axes[0].grid(alpha=0.3)
axes[1].plot(ep_range,history['lr'],lw=2,color='green',marker='o',ms=3)
axes[1].set_title('Learning Rate',fontsize=14,fontweight='bold')
axes[1].set_xlabel('Epoch'); axes[1].set_yscale('log'); axes[1].grid(alpha=0.3)
axes[2].plot(ep_range,history['throughput'],lw=2,color='purple',marker='o',ms=3)
axes[2].axhline(np.mean(history['throughput']),color='red',ls='--',lw=2,
                label=f"Avg: {np.mean(history['throughput']):.0f}")
axes[2].set_title('Throughput (samp/s)',fontsize=14,fontweight='bold')
axes[2].set_xlabel('Epoch'); axes[2].legend(); axes[2].grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f'{MODEL_NAME}_training.png',dpi=150,bbox_inches='tight')
plt.show()
print("✅ Block 13 done")


✅ Block 13 done


In [12]:
# ========================= BLOCK 14: XAI DEPENDENCIES =========================
import subprocess, sys
subprocess.check_call([sys.executable,'-m','pip','install','-q',
                       'lime','shap','grad-cam==1.5.5','imageio[ffmpeg]'])
import lime, lime.lime_image, shap, imageio
from lime.lime_image import LimeImageExplainer
from pytorch_grad_cam import GradCAM, HiResCAM, ShapleyCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from skimage.segmentation import mark_boundaries
print('✅ Block 14 done — XAI packages loaded')



[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


✅ Block 14 done — XAI packages loaded


In [13]:
# ========================= BLOCK 14.5: DATA DEVICE CONFIG =========================
DATA_DEVICE    = "cpu"    # all XAI tensors stored here (system RAM)
COMPUTE_DEVICE = DEVICE   # model + inference run here (GPU VRAM)

# Free any lingering GPU memory before XAI blocks
torch.cuda.empty_cache()
gc.collect()

if torch.cuda.is_available():
    free, total = torch.cuda.mem_get_info()
    print(f"   VRAM free: {free/1e9:.2f} GB / {total/1e9:.2f} GB")

print("✅ Block 14.5 done — XAI data will be stored on CPU, computed on GPU")

   VRAM free: 5.87 GB / 6.09 GB
✅ Block 14.5 done — XAI data will be stored on CPU, computed on GPU


In [14]:
# ========================= BLOCK 15: XAI SETUP =========================
import cv2
from pathlib import Path
import matplotlib.backends.backend_agg as agg

# ── sklearn mock (required by LIME internals) ─────────────────────────────────
import types
from importlib.machinery import ModuleSpec

def _ensure_submod(full, **attrs):
    if full not in sys.modules:
        m=types.ModuleType(full); m.__spec__=ModuleSpec(name=full,loader=None)
        for k,v in attrs.items(): setattr(m,k,v)
        sys.modules[full]=m
        parent='.'.join(full.split('.')[:-1])
        if parent in sys.modules: setattr(sys.modules[parent],full.split('.')[-1],m)

class _LabelEncoder:
    def fit_transform(self,y): self.classes_=np.unique(y); return y
    def transform(self,y): return y

class _MinMaxScaler:
    def __init__(self,**kw): pass
    def fit_transform(self,X): return X
    def transform(self,X): return X

class _Ridge:
    def __init__(self,alpha=1.0,**kw): self.alpha=alpha
    def fit(self,X,y,sample_weight=None):
        W=np.diag(sample_weight) if sample_weight is not None else np.eye(len(y))
        A=X.T@W@X+self.alpha*np.eye(X.shape[1]); b=X.T@W@y
        self.coef_=np.linalg.solve(A,b); self.intercept_=0.0; return self
    def predict(self,X): return X@self.coef_+self.intercept_
    def score(self,X,y):
        p=self.predict(X); ss_res=((y-p)**2).sum(); ss_tot=((y-y.mean())**2).sum()
        return 1-ss_res/(ss_tot+1e-8)

def _check_random_state(seed):
    if seed is None or isinstance(seed,int): return np.random.RandomState(seed)
    return seed

_ensure_submod('sklearn')
_ensure_submod('sklearn.preprocessing',LabelEncoder=_LabelEncoder,MinMaxScaler=_MinMaxScaler)
_ensure_submod('sklearn.linear_model',Ridge=_Ridge)
_ensure_submod('sklearn.utils',check_random_state=_check_random_state)
_ensure_submod('sklearn.utils.validation',check_is_fitted=lambda *a,**k:None)
print('   sklearn mock ✅')

# ── Output tree ────────────────────────────────────────────────────────────────
XAI_ROOT = Path(f'xai_outputs_{MODEL_NAME}')
for sub in ['block_16_gradcam','block_17_lime','block_18_shap',
            'block_19_pdp_ice','block_20_fusions','block_21_dashboard']:
    (XAI_ROOT/sub).mkdir(parents=True,exist_ok=True)
print(f'✅ Output tree: {XAI_ROOT.resolve()}/')

# ── Load checkpoint ────────────────────────────────────────────────────────────
ckpt_path=f'{MODEL_NAME}_best.pt'
if os.path.exists(ckpt_path):
    raw=torch.load(ckpt_path,map_location=DEVICE,weights_only=False)
    state=raw.get('model',raw)
    missing,unexpected=model.load_state_dict(state,strict=False)
    if missing: print(f'   ⚠️  Missing ({len(missing)}): {missing[:3]}')
    print(f'✅ Loaded checkpoint (val={raw.get("val_loss","?")!r})')
else:
    print('⚠️  No checkpoint found — using current weights')
model.eval()

# ── Detect embed dim ──────────────────────────────────────────────────────────
with torch.no_grad():
    _p=torch.zeros(1,3,TARGET_SIZE,TARGET_SIZE,device=DEVICE)
    ACTUAL_EMBED_DIM=model.vit(_p).shape[1]; del _p
torch.cuda.empty_cache()
print(f'✅ Detected embed_dim = {ACTUAL_EMBED_DIM}')

# ── FrameWrapper for CAM ──────────────────────────────────────────────────────
class FrameWrapper(nn.Module):
    def __init__(self,base_model,embed_dim):
        super().__init__()
        self.vit=base_model.vit; self.pool=nn.AdaptiveAvgPool2d(1)
        self.head=nn.Linear(embed_dim,NUM_ACTIONS*ACTION_BINS)
    def forward(self,x): return self.head(self.pool(self.vit(x)).flatten(1))

frame_model=FrameWrapper(model,ACTUAL_EMBED_DIM).to(DEVICE).eval()
with torch.no_grad():
    _t=torch.zeros(1,3,TARGET_SIZE,TARGET_SIZE,device=DEVICE)
    assert frame_model(_t).shape==(1,NUM_ACTIONS*ACTION_BINS); del _t
torch.cuda.empty_cache()
print('✅ FrameWrapper sanity passed')

# ── Target layer (last conv block) ────────────────────────────────────────────
def _get_target_layer(fw):
    all_mods = list(fw.vit.named_modules())
    # 1️⃣ Last BatchNorm2d — ideal for ConvNeXt/CNN-style backbones
    for name, mod in reversed(all_mods):
        if isinstance(mod, nn.BatchNorm2d):
            print(f'   Target layer: {name}  ({mod.__class__.__name__})')
            return [mod]
    # 2️⃣ Last LayerNorm — fallback for transformer-style backbones
    for name, mod in reversed(all_mods):
        if isinstance(mod, nn.LayerNorm):
            print(f'   Target layer: {name}  ({mod.__class__.__name__})')
            return [mod]
    # 3️⃣ Last Conv2d — universal fallback
    for name, mod in reversed(all_mods):
        if isinstance(mod, nn.Conv2d):
            print(f'   Target layer: {name}  ({mod.__class__.__name__})')
            return [mod]
    raise RuntimeError('Could not find a suitable target layer for GradCAM.')
target_layer=_get_target_layer(frame_model)
cam_targets=[ClassifierOutputTarget(128)]

# ── Static XAI_N samples — STORED ON CPU ──────────────────────────────────────
XAI_N=4
xai_videos,xai_insts,xai_toks=[],[],[]
for v,inst,tok,tid in loader_val:
    xai_videos.append(v); xai_insts+=list(inst); xai_toks.append(tok)
    if len(xai_videos)>=XAI_N: break
xai_videos=torch.cat(xai_videos[:XAI_N]).cpu()       # ← CPU!
xai_toks=torch.cat(xai_toks[:XAI_N]).cpu()            # ← CPU!
xai_insts=xai_insts[:XAI_N]
with torch.no_grad():
    embs=[]
    for ins in xai_insts:
        embs.append(text_encoder.encode(ins).cpu())    # ← encode then move to CPU
    xai_text_embs=torch.stack(embs).squeeze(1)         # on CPU
torch.cuda.empty_cache()
print(f'✅ Static batch: video={xai_videos.shape}  text={xai_text_embs.shape}  (stored on CPU)')

# ── 100-frame video sequence — STORED ON CPU ──────────────────────────────────
VID_FRAMES=100; VID_FPS=10
val_tf=transforms.Compose([transforms.ToPILImage(),
    transforms.Resize((TARGET_SIZE,TARGET_SIZE)),transforms.ToTensor()])
all_images_ds=dataset_val.all_images; all_insts_ds=np.array(dataset_val.instructions)
vid_start=SEQ_LEN; vid_end=min(vid_start+VID_FRAMES,len(all_images_ds))
VID_ACTUAL=vid_end-vid_start
VID_INSTRUCTION=str(all_insts_ds[vid_start])[:55]

vid_tensors=[]; vid_numpys=[]
for idx in range(vid_start,vid_end):
    img=all_images_ds[idx]
    if img.dtype!=np.uint8: img=(img*255).astype(np.uint8) if img.max()<=1.0 else img.astype(np.uint8)
    t=val_tf(img).unsqueeze(0).cpu()                   # ← CPU!
    np_f=np.clip(t[0].numpy().transpose(1,2,0),0,1).astype(np.float32)
    vid_tensors.append(t); vid_numpys.append(np_f)
print(f'✅ Video sequence: {VID_ACTUAL} frames | "{VID_INSTRUCTION}" (stored on CPU)')

# ── Shared helpers ─────────────────────────────────────────────────────────────
rt1_tok=RT1Tokenizer()

@torch.no_grad()
def predict_logits(videos,text_embs):
    """Move to GPU, run inference, return on GPU."""
    v=videos.to(DEVICE) if videos.device.type!='cuda' else videos
    t=text_embs.to(DEVICE) if text_embs.device.type!='cuda' else text_embs
    with torch.amp.autocast('cuda',dtype=AUTOCAST_DTYPE): out=model(v,t).float()
    del v,t; torch.cuda.empty_cache()
    return out

def last_frame_np(video_tensor,idx=0,frame=-1):
    return video_tensor[idx,:,frame].cpu().float().permute(1,2,0).numpy().clip(0,1)

def normalise(arr):
    a=arr.astype(np.float32); return (a-a.min())/(a.max()-a.min()+1e-8)

def resize_to(arr_2d,h,w):
    return cv2.resize(arr_2d.astype(np.float32),(w,h),interpolation=cv2.INTER_LINEAR)

def fig_to_rgb(fig):
    fig.canvas.draw()
    # Compatible with newer matplotlib (tostring_rgb removed in 3.8+)
    buf = fig.canvas.buffer_rgba()
    arr = np.asarray(buf)
    return arr[:,:,:3].copy()  # drop alpha channel

def make_header(canvas,text,h_bar=42):
    canvas[:h_bar,:]=22
    cv2.putText(canvas,text,(10,h_bar-12),cv2.FONT_HERSHEY_SIMPLEX,0.52,(220,220,220),1)

def save_video(frames_rgb,path,fps,gif=True):
    h,w=frames_rgb[0].shape[:2]; mp4=str(path)+'.mp4'
    wr=cv2.VideoWriter(mp4,cv2.VideoWriter_fourcc(*'mp4v'),fps,(w,h))
    for f in frames_rgb: wr.write(cv2.cvtColor(f,cv2.COLOR_RGB2BGR))
    wr.release()
    if gif:
        gif_p=str(path)+'.gif'; small=[cv2.resize(f,(w//2,h//2)) for f in frames_rgb]
        imageio.mimsave(gif_p,small,fps=fps,loop=0)
        print(f'      💾 {gif_p}  ({os.path.getsize(gif_p)/1024/1024:.1f} MB)')
    print(f'      💾 {mp4}')

def save_fig(fig,path):
    fig.savefig(str(path),dpi=150,bbox_inches='tight'); plt.show(); plt.close(fig)
    print(f'      💾 {path}')

def block_summary(folder):
    print(f'\n   📁 {folder.name}/')
    for f in sorted(folder.iterdir()):
        kb=f.stat().st_size/1024
        print(f'      📄 {f.name}  '+(f'{kb/1024:.1f} MB' if kb>1024 else f'{kb:.0f} KB'))

# ── Report VRAM ───────────────────────────────────────────────────────────────
if torch.cuda.is_available():
    free, total = torch.cuda.mem_get_info()
    print(f'   VRAM free: {free/1e9:.2f} GB / {total/1e9:.2f} GB')

print('\n✅ Block 15 done — all helpers ready')
print(f'   Run order: 16 → 17 → 18 → 19 → 20 → 21')

   sklearn mock ✅
✅ Output tree: /media/sparsh/CaptainSlow/Programming Stuff/Programming Stuff/Code _n_ Stuff/All Projects/All Projects/VLAs/my work/xai_outputs_rt1_lightweight_bow_convnext/
✅ Loaded checkpoint (val=1.894087577160494)
✅ Detected embed_dim = 192
✅ FrameWrapper sanity passed
   Target layer: final_norm.norm  (LayerNorm)
✅ Static batch: video=torch.Size([16, 3, 6, 224, 224])  text=torch.Size([4, 256])  (stored on CPU)
✅ Video sequence: 100 frames | "route cable" (stored on CPU)
   VRAM free: 5.86 GB / 6.09 GB

✅ Block 15 done — all helpers ready
   Run order: 16 → 17 → 18 → 19 → 20 → 21


In [15]:
# ========================= BLOCK 16: GRADCAM FAMILY =========================
# GradCAM | HiResCAM | ShapleyCAM
# Outputs: static, side-by-side video, filmstrip, evolution, trajectory

DIR16=XAI_ROOT/'block_16_gradcam'
CAM_METHODS={'GradCAM':GradCAM,'HiResCAM':HiResCAM,'ShapleyCAM':ShapleyCAM}
CAM_COLORS={'GradCAM':'royalblue','HiResCAM':'seagreen','ShapleyCAM':'crimson'}
FILMSTRIP_N=10

print('='*70); print('🔥 BLOCK 16: GradCAM Family'); print('='*70)

if 'target_layer' not in dir() or target_layer is None:
    print('⚠️  target_layer not found — re-deriving...')
    def _get_target_layer(fw):
        all_mods = list(fw.vit.named_modules())
        for name, mod in reversed(all_mods):
            if isinstance(mod, nn.BatchNorm2d):
                print(f'   Target layer: {name}  ({mod.__class__.__name__})'); return [mod]
        for name, mod in reversed(all_mods):
            if isinstance(mod, nn.Conv2d):
                print(f'   Target layer: {name}  ({mod.__class__.__name__})'); return [mod]
        raise RuntimeError('No suitable GradCAM target layer found.')
    target_layer = _get_target_layer(frame_model)
    cam_targets  = [ClassifierOutputTarget(128)]

# ── 16-A  Static CAM ──────────────────────────────────────────────────────────
print('\n[1/6] Static CAM on XAI_N samples...')
cam_results={}
for name,CamCls in CAM_METHODS.items():
    overlays=[]
    with CamCls(model=frame_model,target_layers=target_layer) as cam:
        for i in range(XAI_N):
            frame_np=last_frame_np(xai_videos,i)
            frame_t=xai_videos[i,:,-1].unsqueeze(0)
            gray=cam(input_tensor=frame_t,targets=cam_targets)[0]
            ov=show_cam_on_image(frame_np,gray,use_rgb=True)
            overlays.append((frame_np,gray,ov))
    cam_results[name]=overlays; print(f'   ✅ {name}')

fig,axes=plt.subplots(XAI_N,len(CAM_METHODS)+1,figsize=(5*(len(CAM_METHODS)+1),5*XAI_N))
for i in range(XAI_N):
    axes[i,0].imshow(last_frame_np(xai_videos,i))
    axes[i,0].set_title(f'Original\n"{xai_insts[i][:28]}"',fontsize=8); axes[i,0].axis('off')
    for j,name in enumerate(CAM_METHODS):
        _,_,ov=cam_results[name][i]
        axes[i,j+1].imshow(ov); axes[i,j+1].set_title(name,fontsize=9,fontweight='bold')
        axes[i,j+1].axis('off')
plt.suptitle(f'GradCAM Family — {MODEL_NAME}',fontsize=13,fontweight='bold')
plt.tight_layout(); save_fig(fig,DIR16/'gradcam_static.png')

# ── 16-B  CAM on 100 frames ───────────────────────────────────────────────────
print(f'\n[2/6] Computing CAM on {VID_ACTUAL} frames...')
all_cam_data={}
for name,CamCls in CAM_METHODS.items():
    grays,overlays=[],[]
    print(f'   {name}:',end=' ',flush=True)
    with CamCls(model=frame_model,target_layers=target_layer) as cam:
        for f_idx in range(VID_ACTUAL):
            gray=cam(input_tensor=vid_tensors[f_idx],targets=cam_targets)[0]
            ov=show_cam_on_image(vid_numpys[f_idx],gray,use_rgb=True)
            grays.append(gray); overlays.append(ov)
            if (f_idx+1)%25==0: print(f_idx+1,end=' ',flush=True)
    all_cam_data[name]={'grays':grays,'overlays':overlays}; print('✅')
torch.cuda.empty_cache()

# ── 16-C  Side-by-side video ──────────────────────────────────────────────────
print(f'\n[3/6] Stitching side-by-side video...')
H,W=TARGET_SIZE,TARGET_SIZE; HBAR=42; n_panels=1+len(CAM_METHODS)
cw,ch=W*n_panels,H+HBAR; vid_frames=[]
for f_idx in range(VID_ACTUAL):
    canvas=np.zeros((ch,cw,3),dtype=np.uint8)
    make_header(canvas,f"GradCAM | {f_idx+1}/{VID_ACTUAL} | '{VID_INSTRUCTION}'")
    orig=(vid_numpys[f_idx]*255).astype(np.uint8)
    canvas[HBAR:,0:W]=orig
    cv2.putText(canvas,'Original',(4,HBAR+16),cv2.FONT_HERSHEY_SIMPLEX,0.44,(255,255,255),1)
    for m_idx,name in enumerate(CAM_METHODS):
        x0=(m_idx+1)*W; canvas[HBAR:,x0:x0+W]=all_cam_data[name]['overlays'][f_idx]
        cv2.putText(canvas,name,(x0+4,HBAR+16),cv2.FONT_HERSHEY_SIMPLEX,0.44,(255,255,255),1)
    vid_frames.append(canvas)
save_video(vid_frames,DIR16/'gradcam_sidebyside',VID_FPS)

# ── 16-D  Filmstrip ───────────────────────────────────────────────────────────
print('\n[4/6] Filmstrip...')
strip_idxs=list(range(0,VID_ACTUAL,FILMSTRIP_N)); n_strip=len(strip_idxs)
fig,axes=plt.subplots(1+len(CAM_METHODS),n_strip,figsize=(3*n_strip,3*(1+len(CAM_METHODS))))
if n_strip==1: axes=axes[:,np.newaxis]
for col,f_idx in enumerate(strip_idxs):
    axes[0,col].imshow(vid_numpys[f_idx]); axes[0,col].set_title(f'F{f_idx+1}',fontsize=7)
    axes[0,col].axis('off')
    if col==0: axes[0,col].set_ylabel('Original',fontsize=9,fontweight='bold')
    for row,name in enumerate(CAM_METHODS,start=1):
        axes[row,col].imshow(all_cam_data[name]['overlays'][f_idx]); axes[row,col].axis('off')
        if col==0: axes[row,col].set_ylabel(name,fontsize=9,fontweight='bold')
plt.suptitle(f'GradCAM Filmstrip (every {FILMSTRIP_N} frames)\n"{VID_INSTRUCTION}"',fontsize=12,fontweight='bold')
plt.tight_layout(); save_fig(fig,DIR16/'gradcam_filmstrip.png')

# ── 16-E  Attention evolution ─────────────────────────────────────────────────
print('\n[5/6] Attention evolution...')
attn_stats={m:{'cx':[],'cy':[],'spread':[],'peak':[]} for m in CAM_METHODS}
for name in CAM_METHODS:
    for f_idx in range(VID_ACTUAL):
        gray=all_cam_data[name]['grays'][f_idx]; hc,wc=gray.shape; total=gray.sum()+1e-8
        yy,xx=np.mgrid[0:hc,0:wc]
        cx=(xx*gray).sum()/total; cy=(yy*gray).sum()/total
        spread=np.sqrt((((xx-cx)**2+(yy-cy)**2)*gray).sum()/total)
        attn_stats[name]['cx'].append(cx/wc); attn_stats[name]['cy'].append(cy/hc)
        attn_stats[name]['spread'].append(spread/max(hc,wc)); attn_stats[name]['peak'].append(gray.max())
frames_ax=np.arange(1,VID_ACTUAL+1)
fig,axes=plt.subplots(2,2,figsize=(16,9))
metrics=[('cx','Attention Center X','Norm. X'),('cy','Attention Center Y','Norm. Y'),
         ('spread','Attention Spread σ','Norm. spread'),('peak','Peak Activation','Max CAM')]
for ax,(key,title,ylabel) in zip(axes.flat,metrics):
    for name in CAM_METHODS:
        ax.plot(frames_ax,attn_stats[name][key],color=CAM_COLORS[name],label=name,lw=1.6,alpha=0.85)
    ax.set_title(title,fontweight='bold'); ax.set_xlabel('Frame'); ax.set_ylabel(ylabel)
    ax.legend(); ax.grid(alpha=0.3)
plt.suptitle(f'Temporal Attention Evolution — "{VID_INSTRUCTION}"',fontsize=13,fontweight='bold')
plt.tight_layout(); save_fig(fig,DIR16/'gradcam_evolution.png')

# ── 16-F  Trajectory map ──────────────────────────────────────────────────────
print('\n[6/6] Attention trajectory...')
fig,axes=plt.subplots(1,len(CAM_METHODS),figsize=(6*len(CAM_METHODS),6))
mid_frame=vid_numpys[VID_ACTUAL//2]; time_cols=plt.cm.coolwarm(np.linspace(0,1,VID_ACTUAL))
for ax,name in zip(axes,CAM_METHODS):
    cx=np.array(attn_stats[name]['cx'])*TARGET_SIZE; cy=np.array(attn_stats[name]['cy'])*TARGET_SIZE
    sp=np.array(attn_stats[name]['spread'])*TARGET_SIZE
    ax.imshow(mid_frame); ax.plot(cx,cy,color='white',lw=1.2,alpha=0.35,zorder=1)
    for f_idx in range(VID_ACTUAL):
        ax.scatter(cx[f_idx],cy[f_idx],c=[time_cols[f_idx]],s=max(8,sp[f_idx]*3),
                   alpha=0.7,edgecolors='white',linewidths=0.25,zorder=2)
    ax.scatter(cx[0],cy[0],c='cyan',s=140,marker='^',edgecolors='white',lw=2,zorder=3,label='Start')
    ax.scatter(cx[-1],cy[-1],c='magenta',s=140,marker='v',edgecolors='white',lw=2,zorder=3,label='End')
    ax.set_title(f'{name} — Trajectory',fontsize=12,fontweight='bold')
    ax.legend(fontsize=9); ax.axis('off')
plt.suptitle(f'Attention Path  blue=early → red=late  size∝spread\n"{VID_INSTRUCTION}"',fontsize=12,fontweight='bold')
plt.tight_layout(); save_fig(fig,DIR16/'gradcam_trajectory.png')

block_summary(DIR16); print('\n✅ Block 16 done')


🔥 BLOCK 16: GradCAM Family

[1/6] Static CAM on XAI_N samples...
   ✅ GradCAM
   ✅ HiResCAM
   ✅ ShapleyCAM
      💾 xai_outputs_rt1_lightweight_bow_convnext/block_16_gradcam/gradcam_static.png

[2/6] Computing CAM on 100 frames...
   GradCAM: 25 50 75 100 ✅
   HiResCAM: 25 50 75 100 ✅
   ShapleyCAM: 25 50 75 100 ✅

[3/6] Stitching side-by-side video...
      💾 xai_outputs_rt1_lightweight_bow_convnext/block_16_gradcam/gradcam_sidebyside.gif  (3.5 MB)
      💾 xai_outputs_rt1_lightweight_bow_convnext/block_16_gradcam/gradcam_sidebyside.mp4

[4/6] Filmstrip...
      💾 xai_outputs_rt1_lightweight_bow_convnext/block_16_gradcam/gradcam_filmstrip.png

[5/6] Attention evolution...
      💾 xai_outputs_rt1_lightweight_bow_convnext/block_16_gradcam/gradcam_evolution.png

[6/6] Attention trajectory...
      💾 xai_outputs_rt1_lightweight_bow_convnext/block_16_gradcam/gradcam_trajectory.png

   📁 block_16_gradcam/
      📄 gradcam_evolution.png  182 KB
      📄 gradcam_filmstrip.png  6.2 MB
      📄 gra

In [17]:
# ========================= BLOCK 17: LIME =========================
DIR17=XAI_ROOT/'block_17_lime'
LIME_SAMPLES_STATIC=128; LIME_SAMPLES_VIDEO=50; LIME_TOP_K=5; FILMSTRIP_N=10

print('='*70); print('🍋 BLOCK 17: LIME'); print('='*70)

def lime_predict(images_np):
    tensors=[torch.from_numpy(img.astype(np.float32)).permute(2,0,1) for img in images_np]
    batch=torch.stack(tensors).to(DEVICE)
    video=batch.unsqueeze(2).expand(-1,-1,SEQ_LEN,-1,-1)                  # already on GPU
    text_emb=xai_text_embs[0:1].to(DEVICE).expand(len(images_np),-1)     # ← .to(DEVICE) added
    with torch.no_grad(), torch.amp.autocast('cuda',dtype=AUTOCAST_DTYPE):
        logits=model(video,text_emb).float()
    bins=torch.arange(256,device=DEVICE).float(); probs=torch.softmax(logits,dim=-1)
    del video, text_emb; torch.cuda.empty_cache()                          # ← cleanup added
    return ((probs*bins[None,None,:]).sum(-1)/255.0).cpu().numpy()

def lime_overlay(frame_np,mask):
    mask_2d=mask[:,:,0] if mask.ndim==3 else mask; mask_2d=(mask_2d==1)
    ov=frame_np.copy(); green=np.array([0.,0.85,0.2],dtype=np.float32)
    ov[mask_2d]=ov[mask_2d]*0.35+green*0.65
    return np.clip(ov,0,1)

explainer_lime=LimeImageExplainer()

# ── 17-A  Static LIME ─────────────────────────────────────────────────────────
print(f'\n[1/5] Static LIME ({LIME_SAMPLES_STATIC} samples × {XAI_N} frames)...')
lime_results=[]
for i in range(XAI_N):
    frame_np=last_frame_np(xai_videos,i)
    exp=explainer_lime.explain_instance(frame_np.astype(np.float64),lime_predict,
        top_labels=7,hide_color=0,num_samples=LIME_SAMPLES_STATIC,num_features=20)
    _,mask=exp.get_image_and_mask(label=0,positive_only=True,num_features=LIME_TOP_K,hide_rest=False)
    mask_2d=mask[:,:,0] if mask.ndim==3 else mask
    lime_results.append((frame_np,mask_2d,exp))
    print(f'   sample {i+1}/{XAI_N} done')

fig,axes=plt.subplots(2,XAI_N,figsize=(5*XAI_N,10))
for i,(frame_np,mask_2d,_) in enumerate(lime_results):
    axes[0,i].imshow(frame_np); axes[0,i].set_title(f'Sample {i+1}\n"{xai_insts[i][:25]}"',fontsize=8)
    axes[0,i].axis('off')
    axes[1,i].imshow(lime_overlay(frame_np,mask_2d))
    axes[1,i].set_title(f'Top-{LIME_TOP_K} superpixels',fontsize=8); axes[1,i].axis('off')
plt.suptitle(f'LIME — {MODEL_NAME}',fontsize=13,fontweight='bold')
plt.tight_layout(); save_fig(fig,DIR17/'lime_static.png')

# ── 17-B  LIME video ──────────────────────────────────────────────────────────
print(f'\n[2/5] LIME video ({LIME_SAMPLES_VIDEO} samples × {VID_ACTUAL} frames)...')
lime_masks=[]; lime_overlays=[]; active_area=[]; t0=time.time()
for f_idx in range(VID_ACTUAL):
    frame_np=vid_numpys[f_idx]
    exp=explainer_lime.explain_instance(frame_np.astype(np.float64),lime_predict,
        top_labels=7,hide_color=0,num_samples=LIME_SAMPLES_VIDEO,num_features=20)
    _,mask=exp.get_image_and_mask(label=0,positive_only=True,num_features=LIME_TOP_K,hide_rest=False)
    mask_2d=mask[:,:,0] if mask.ndim==3 else mask
    ov_float=lime_overlay(frame_np,mask_2d)
    lime_masks.append(mask_2d); lime_overlays.append((ov_float*255).astype(np.uint8))
    active_area.append(mask_2d.mean())
    if (f_idx+1)%10==0:
        elapsed=time.time()-t0; eta=elapsed/(f_idx+1)*(VID_ACTUAL-f_idx-1)
        print(f'   [{f_idx+1:3d}/{VID_ACTUAL}]  active={mask_2d.mean():.3f}  elapsed={elapsed:.0f}s  ETA={eta:.0f}s')
print(f'   ✅ LIME done in {time.time()-t0:.0f}s')

# ── 17-C  Stitch LIME video ───────────────────────────────────────────────────
print('\n[3/5] Stitching LIME video...')
H,W=TARGET_SIZE,TARGET_SIZE; HBAR=42; cw,ch=W*2,H+HBAR; frames_rgb=[]
for f_idx in range(VID_ACTUAL):
    canvas=np.zeros((ch,cw,3),dtype=np.uint8)
    make_header(canvas,f"LIME | {f_idx+1}/{VID_ACTUAL} | active={active_area[f_idx]:.1%} | '{VID_INSTRUCTION}'")
    canvas[HBAR:,0:W]=(vid_numpys[f_idx]*255).astype(np.uint8)
    cv2.putText(canvas,'Original',(4,HBAR+16),cv2.FONT_HERSHEY_SIMPLEX,0.44,(255,255,255),1)
    canvas[HBAR:,W:2*W]=lime_overlays[f_idx]
    cv2.putText(canvas,f'LIME top-{LIME_TOP_K}',(W+4,HBAR+16),cv2.FONT_HERSHEY_SIMPLEX,0.44,(255,255,255),1)
    frames_rgb.append(canvas)
save_video(frames_rgb,DIR17/'lime_video',VID_FPS)

# ── 17-D  Filmstrip ───────────────────────────────────────────────────────────
print('\n[4/5] Filmstrip...')
strip_idxs=list(range(0,VID_ACTUAL,FILMSTRIP_N)); n_strip=len(strip_idxs)
fig,axes=plt.subplots(2,n_strip,figsize=(3*n_strip,6))
if n_strip==1: axes=axes[:,np.newaxis]
for col,f_idx in enumerate(strip_idxs):
    axes[0,col].imshow(vid_numpys[f_idx]); axes[0,col].set_title(f'F{f_idx+1}',fontsize=7)
    axes[0,col].axis('off')
    if col==0: axes[0,col].set_ylabel('Original',fontsize=9,fontweight='bold')
    axes[1,col].imshow(lime_overlays[f_idx]); axes[1,col].axis('off')
    if col==0: axes[1,col].set_ylabel('LIME',fontsize=9,fontweight='bold')
plt.suptitle(f'LIME Filmstrip (every {FILMSTRIP_N} frames)\n"{VID_INSTRUCTION}"',fontsize=12,fontweight='bold')
plt.tight_layout(); save_fig(fig,DIR17/'lime_filmstrip.png')

# ── 17-E  Activity analysis ───────────────────────────────────────────────────
print('\n[5/5] Activity analysis...')
active_arr=np.array(active_area); mask_stack=np.stack(lime_masks); mean_mask=mask_stack.mean(axis=0)
def rolling(arr,w=10): return np.convolve(arr,np.ones(w)/w,mode='same')
fig,axes=plt.subplots(1,3,figsize=(18,5))
frames_ax=np.arange(1,VID_ACTUAL+1)
axes[0].fill_between(frames_ax,active_arr,alpha=0.25,color='limegreen')
axes[0].plot(frames_ax,active_arr,color='green',lw=1.2,alpha=0.55,label='Per-frame')
axes[0].plot(frames_ax,rolling(active_arr),color='darkgreen',lw=2.2,label='Rolling mean')
axes[0].set_title('Highlighted Region Size',fontweight='bold'); axes[0].set_xlabel('Frame')
axes[0].set_ylabel('Fraction highlighted'); axes[0].legend(); axes[0].grid(alpha=0.3)
mid=vid_numpys[VID_ACTUAL//2]
axes[1].imshow(mid); axes[1].imshow(mean_mask,cmap='Greens',alpha=0.6)
axes[1].set_title('Aggregate LIME Attention\n(mean mask)',fontweight='bold'); axes[1].axis('off')
axes[2].imshow(mean_mask,cmap='hot'); axes[2].set_title('Mean Superpixel Heatmap',fontweight='bold')
axes[2].axis('off')
plt.suptitle(f'LIME Temporal Analysis — "{VID_INSTRUCTION}"',fontsize=13,fontweight='bold')
plt.tight_layout(); save_fig(fig,DIR17/'lime_activity.png')

block_summary(DIR17); print('\n✅ Block 17 done')


🍋 BLOCK 17: LIME

[1/5] Static LIME (128 samples × 4 frames)...


  0%|          | 0/128 [00:00<?, ?it/s]

   sample 1/4 done


  0%|          | 0/128 [00:00<?, ?it/s]

   sample 2/4 done


...

  0%|          | 0/50 [00:00<?, ?it/s]

  0%|          | 0/50 [00:00<?, ?it/s]

  0%|          | 0/50 [00:00<?, ?it/s]

  0%|          | 0/50 [00:00<?, ?it/s]

   [100/100]  active=0.137  elapsed=58s  ETA=0s
   ✅ LIME done in 58s

[3/5] Stitching LIME video...
      💾 xai_outputs_rt1_lightweight_bow_convnext/block_17_lime/lime_video.gif  (2.0 MB)
      💾 xai_outputs_rt1_lightweight_bow_convnext/block_17_lime/lime_video.mp4

[4/5] Filmstrip...
      💾 xai_outputs_rt1_lightweight_bow_convnext/block_17_lime/lime_filmstrip.png

[5/5] Activity analysis...
      💾 xai_outputs_rt1_lightweight_bow_convnext/block_17_lime/lime_activity.png

   📁 block_17_lime/
      📄 lime_activity.png  684 KB
      📄 lime_filmstrip.png  3.3 MB
      📄 lime_static.png  890 KB
      📄 lime_video.gif  2.0 MB
      📄 lime_video.mp4  1.3 MB

✅ Block 17 done


In [18]:
# ========================= BLOCK 18: SHAP =========================
# Uses Gradient×Input attribution instead of GradientExplainer.
# Equivalent spatial visualisation; requires only 1 forward+backward
# pass per frame, so it fits in < 2 GB VRAM.
DIR18=XAI_ROOT/'block_18_shap'
print('='*70); print('🔵 BLOCK 18: SHAP (Grad×Input — memory-safe)'); print('='*70)

# ── Reuse FrameWrapper already on GPU ────────────────────────────────────────
#    frame_model shares weights with model — no extra VRAM needed
torch.cuda.empty_cache(); gc.collect()
if torch.cuda.is_available():
    free,total=torch.cuda.mem_get_info()
    print(f'   VRAM free at SHAP start: {free/1e9:.2f}/{total/1e9:.2f} GB')

def grad_x_input(inp_cpu):
    """Return H×W attribution map via |gradient × input|, one fwd+bwd pass."""
    inp = inp_cpu.clone().to(DEVICE).requires_grad_(True)
    with torch.amp.autocast('cuda', dtype=AUTOCAST_DTYPE):
        out = frame_model(inp)          # (1, NUM_ACTIONS*ACTION_BINS)
    out.sum().backward()
    # attr shape: (C,H,W) → collapse channels → (H,W)
    attr = (inp.grad.detach() * inp.detach()).abs().mean(dim=1)[0].cpu().numpy()
    inp.requires_grad_(False)
    del inp, out
    torch.cuda.empty_cache()
    return attr

# ── 18-A  Static SHAP ─────────────────────────────────────────────────────────
print(f'\n[1/5] Static attribution ({XAI_N} samples)...')
shap_agg_vis=[]
for i in range(XAI_N):
    torch.cuda.empty_cache(); gc.collect()
    single = xai_videos[i:i+1,:,-1]        # last frame, on CPU
    sv_map = grad_x_input(single)
    shap_agg_vis.append(sv_map)
    print(f'   sample {i+1}/{XAI_N} done')
shap_agg_vis=np.stack(shap_agg_vis)
torch.cuda.empty_cache()

fig,axes=plt.subplots(2,XAI_N,figsize=(5*XAI_N,10))
for i in range(XAI_N):
    frame_np=last_frame_np(xai_videos,i)
    axes[0,i].imshow(frame_np); axes[0,i].set_title(f'Input Frame {i+1}',fontsize=9); axes[0,i].axis('off')
    smap=shap_agg_vis[i]; smap=(smap-smap.min())/(smap.max()-smap.min()+1e-8)
    im=axes[1,i].imshow(smap,cmap='hot',vmin=0,vmax=1)
    axes[1,i].set_title('|Grad×Input|',fontsize=8); axes[1,i].axis('off')
    plt.colorbar(im,ax=axes[1,i],fraction=0.046)
plt.suptitle(f'SHAP (Grad×Input) — {MODEL_NAME}',fontsize=13,fontweight='bold')
plt.tight_layout(); save_fig(fig,DIR18/'shap_static.png')

# ── 18-B  Attribution video (1 fwd+bwd per frame — no chunking needed) ───────
print(f'\n[2/5] Attribution video ({VID_ACTUAL} frames)...')
cmap_hot=plt.cm.hot; shap_vid_maps=[]; t0=time.time()
for f_idx in range(VID_ACTUAL):
    sv_map = grad_x_input(vid_tensors[f_idx])
    shap_vid_maps.append(sv_map.copy())
    if (f_idx+1)%20==0:
        elapsed=time.time()-t0; eta=elapsed/(f_idx+1)*(VID_ACTUAL-f_idx-1)
        print(f'   [{f_idx+1:3d}/{VID_ACTUAL}]  elapsed={elapsed:.0f}s  ETA={eta:.0f}s')
torch.cuda.empty_cache(); gc.collect()
print(f'   ✅ done in {time.time()-t0:.0f}s')
all_shap_arr=np.stack(shap_vid_maps); global_max=all_shap_arr.max()+1e-8
shap_norm_maps=all_shap_arr/global_max

def shap_blend(frame_np,shap_norm):
    heat=cmap_hot(shap_norm)[:,:,:3].astype(np.float32)
    return np.clip(frame_np*0.45+heat*0.55,0,1)

# ── 18-C  Stitch SHAP video ───────────────────────────────────────────────────
print('\n[3/5] Stitching attribution video...')
H,W=TARGET_SIZE,TARGET_SIZE; HBAR=42; cw,ch=W*3,H+HBAR; frames_rgb=[]
for f_idx in range(VID_ACTUAL):
    frame_np=vid_numpys[f_idx]; shap_norm=shap_norm_maps[f_idx]
    heat=cmap_hot(shap_norm)[:,:,:3]; blend=shap_blend(frame_np,shap_norm)
    canvas=np.zeros((ch,cw,3),dtype=np.uint8)
    make_header(canvas,f"SHAP | {f_idx+1}/{VID_ACTUAL} | '{VID_INSTRUCTION}'")
    canvas[HBAR:,0:W]=(frame_np*255).astype(np.uint8)
    canvas[HBAR:,W:2*W]=(heat*255).astype(np.uint8)
    canvas[HBAR:,2*W:3*W]=(blend*255).astype(np.uint8)
    for lbl,x0 in [('Original',0),('|Grad×Input| heat',W),('Overlay',2*W)]:
        cv2.putText(canvas,lbl,(x0+4,HBAR+16),cv2.FONT_HERSHEY_SIMPLEX,0.44,(255,255,255),1)
    frames_rgb.append(canvas)
save_video(frames_rgb,DIR18/'shap_video',VID_FPS)

# ── 18-D  Filmstrip ───────────────────────────────────────────────────────────
print('\n[4/5] Filmstrip...')
strip_idxs=list(range(0,VID_ACTUAL,10)); n_strip=len(strip_idxs)
fig,axes=plt.subplots(3,n_strip,figsize=(3*n_strip,9))
if n_strip==1: axes=axes[:,np.newaxis]
for col,f_idx in enumerate(strip_idxs):
    frame_np=vid_numpys[f_idx]; shap_norm=shap_norm_maps[f_idx]; blend=shap_blend(frame_np,shap_norm)
    for row,(img,lbl,kw) in enumerate([(frame_np,'Original',{}),(shap_norm,'|Grad×Input|',{'cmap':'hot','vmin':0,'vmax':1}),(blend,'Overlay',{})]):
        axes[row,col].imshow(img,**kw); axes[row,col].set_title(f'F{f_idx+1}',fontsize=7)
        axes[row,col].axis('off')
        if col==0: axes[row,col].set_ylabel(lbl,fontsize=9,fontweight='bold')
plt.suptitle(f'Grad×Input Filmstrip — "{VID_INSTRUCTION}"',fontsize=12,fontweight='bold')
plt.tight_layout(); save_fig(fig,DIR18/'shap_filmstrip.png')

# ── 18-E  Mean attribution ────────────────────────────────────────────────────
print('\n[5/5] Mean attribution...')
mean_shap=shap_norm_maps.mean(axis=0); mid=vid_numpys[VID_ACTUAL//2]
fig,axes=plt.subplots(1,3,figsize=(15,5))
axes[0].imshow(mid); axes[0].set_title('Middle Frame',fontweight='bold'); axes[0].axis('off')
axes[1].imshow(mean_shap,cmap='hot',vmin=0,vmax=1)
axes[1].set_title('Mean |Grad×Input| — 100 frames',fontweight='bold'); axes[1].axis('off')
axes[2].imshow(shap_blend(mid,mean_shap))
axes[2].set_title('Mean Attribution Overlay',fontweight='bold'); axes[2].axis('off')
plt.suptitle(f'Aggregate Grad×Input Attribution — "{VID_INSTRUCTION}"',fontsize=13,fontweight='bold')
plt.tight_layout(); save_fig(fig,DIR18/'shap_mean.png')

# ── Free GPU before PDP block ──────────────────────────────────────────────────
torch.cuda.empty_cache(); gc.collect()
if torch.cuda.is_available():
    free,total=torch.cuda.mem_get_info()
    print(f'   VRAM free after SHAP: {free/1e9:.2f}/{total/1e9:.2f} GB')

block_summary(DIR18); print('\n✅ Block 18 done')


🔵 BLOCK 18: SHAP (Grad×Input — memory-safe)
   VRAM free at SHAP start: 5.67/6.09 GB

[1/5] Static attribution (4 samples)...
   sample 1/4 done
   sample 2/4 done
   sample 3/4 done
   sample 4/4 done
      💾 xai_outputs_rt1_lightweight_bow_convnext/block_18_shap/shap_static.png

[2/5] Attribution video (100 frames)...
   [ 20/100]  elapsed=0s  ETA=1s
   [ 40/100]  elapsed=0s  ETA=1s
   [ 60/100]  elapsed=1s  ETA=0s
   [ 80/100]  elapsed=1s  ETA=0s
   [100/100]  elapsed=1s  ETA=0s
   ✅ done in 1s

[3/5] Stitching attribution video...
      💾 xai_outputs_rt1_lightweight_bow_convnext/block_18_shap/shap_video.gif  (3.4 MB)
      💾 xai_outputs_rt1_lightweight_bow_convnext/block_18_shap/shap_video.mp4

[4/5] Filmstrip...
      💾 xai_outputs_rt1_lightweight_bow_convnext/block_18_shap/shap_filmstrip.png

[5/5] Mean attribution...
      💾 xai_outputs_rt1_lightweight_bow_convnext/block_18_shap/shap_mean.png
   VRAM free after SHAP: 5.48/6.09 GB

   📁 block_18_shap/
      📄 shap_filmstrip.png  

In [19]:
# ========================= BLOCK 19: PDP + ICE + VALIDATION =========================
# ── Flush GPU before starting ─────────────────────────────────────────────────
torch.cuda.empty_cache(); gc.collect()
if torch.cuda.is_available():
    _free,_total=torch.cuda.mem_get_info()
    print(f'   VRAM free at PDP start: {_free/1e9:.2f}/{_total/1e9:.2f} GB')
DIR19=XAI_ROOT/'block_19_pdp_ice'
ACTION_NAMES=['wx','wy','wz','rx','ry','rz','term'][:NUM_ACTIONS]
ACTION_COLORS=plt.cm.tab10(np.linspace(0,1,NUM_ACTIONS))
ALPHA_GRID=np.linspace(0.2,1.8,20); PDP_FPS=6; HOLD_FRAMES=12

print('='*70); print('📊 BLOCK 19: PDP + ICE + Validation Improvement'); print('='*70)

# ── 19-A  Validation improvement ─────────────────────────────────────────────
torch.cuda.empty_cache()
def evaluate_val_loss_safe(mdl,n_batches=20):
    mdl.eval(); total,count=0.0,0; loss_fn=nn.CrossEntropyLoss()
    with torch.no_grad():
        for k,(v,inst,tok,tid) in enumerate(loader_val):
            if k>=n_batches: break
            for b in range(v.shape[0]):
                v_s=v[b:b+1].to(DEVICE); tok_s=tok[b:b+1].to(DEVICE)
                emb_s=text_encoder.encode([inst[b]])
                with torch.amp.autocast('cuda',dtype=AUTOCAST_DTYPE):
                    logits=mdl(v_s,emb_s).float()
                B,A,BIN=logits.shape
                total+=loss_fn(logits.view(B*A,BIN),tok_s.view(B*A).long()).item(); count+=1
                del v_s,tok_s,emb_s,logits
                torch.cuda.empty_cache()   # per-sample flush
            gc.collect()
    return total/max(count,1)

print('\n[1/5] Evaluating trained model (n=20 batches)...')
current_val_loss=evaluate_val_loss_safe(model,n_batches=20); torch.cuda.empty_cache()
baseline_val_loss=math.log(256)  # theoretical random
improvement=baseline_val_loss-current_val_loss; pct=improvement/baseline_val_loss*100
improved=improvement>0
print(f'   Baseline (ln 256) : {baseline_val_loss:.4f}')
print(f'   Trained model     : {current_val_loss:.4f}')
print(f'   Δ                 : {improvement:+.4f}  ({pct:+.1f}%)')
print(f'   Result            : {"✅ IMPROVED" if improved else "❌ NOT improved"}')

fig,ax=plt.subplots(figsize=(5,4))
bars=ax.bar(['Random\n(theoretical)','Trained\nModel'],[baseline_val_loss,current_val_loss],
            color=['#e05252','#52e0a0'],width=0.45,edgecolor='black')
for b in bars:
    ax.text(b.get_x()+b.get_width()/2,b.get_height()+0.02,f'{b.get_height():.4f}',
            ha='center',va='bottom',fontsize=11,fontweight='bold')
ax.set_ylabel('Validation Loss (CrossEntropy)'); ax.grid(axis='y',alpha=0.3)
ax.set_title(f'Δ={improvement:+.4f} ({pct:+.1f}%)  {"✅ Improved" if improved else "❌ Not improved"}',fontweight='bold')
plt.tight_layout(); save_fig(fig,DIR19/'validation_improvement.png')

# ── 19-B  PDP + ICE ───────────────────────────────────────────────────────────
print('[2/5] Computing PDP + ICE curves...')
torch.cuda.empty_cache()
def action_score(video,text_emb,alpha):
    v=(video*alpha).clamp(0,1).to(DEVICE)            # CPU→GPU
    te=text_emb.to(DEVICE) if text_emb.device.type!='cuda' else text_emb
    with torch.no_grad(), torch.amp.autocast('cuda',dtype=AUTOCAST_DTYPE):
        logits=model(v,te).float()
    bins=torch.arange(256,device=DEVICE).float(); probs=torch.softmax(logits[0],dim=-1)
    score=((probs*bins.unsqueeze(0)).sum(-1)/255.0).cpu().numpy()
    del logits,probs,v,te; torch.cuda.empty_cache(); return score

pdp_curves=np.zeros((len(ALPHA_GRID),NUM_ACTIONS)); ice_curves=np.zeros((XAI_N,len(ALPHA_GRID),NUM_ACTIONS))
for ai,alpha in enumerate(ALPHA_GRID):
    for si in range(XAI_N):
        score=action_score(xai_videos[si:si+1],xai_text_embs[si:si+1],alpha)
        ice_curves[si,ai]=score; pdp_curves[ai]+=score/XAI_N
    if (ai+1)%5==0: print(f'   α={alpha:.2f}  ({ai+1}/{len(ALPHA_GRID)})')
print('   ✅ done')

# ── 19-C  Static PDP + ICE plot ───────────────────────────────────────────────
print('\n[3/5] Static PDP + ICE plot...')
fig,axes=plt.subplots(2,NUM_ACTIONS,figsize=(4*NUM_ACTIONS,8))
for a in range(NUM_ACTIONS):
    ax=axes[0,a]
    for si in range(XAI_N): ax.plot(ALPHA_GRID,ice_curves[si,:,a],alpha=0.4,lw=1.2,color=ACTION_COLORS[a])
    ax.set_title(f'ICE — {ACTION_NAMES[a]}',fontsize=9,fontweight='bold')
    ax.set_xlabel('α'); ax.set_ylabel('Pred. token'); ax.grid(alpha=0.3)
    ax2=axes[1,a]; ax2.plot(ALPHA_GRID,pdp_curves[:,a],lw=2.5,color='crimson',label='PDP')
    ax2.fill_between(ALPHA_GRID,ice_curves[:,:,a].min(0),ice_curves[:,:,a].max(0),alpha=0.15,color='crimson',label='ICE range')
    ax2.set_title(f'PDP — {ACTION_NAMES[a]}',fontsize=9,fontweight='bold')
    ax2.set_xlabel('α'); ax2.legend(fontsize=7); ax2.grid(alpha=0.3)
plt.suptitle(f'PDP + ICE — Feature: brightness multiplier α  |  {MODEL_NAME}',fontsize=12,fontweight='bold')
plt.tight_layout(); save_fig(fig,DIR19/'pdp_ice_static.png')

# ── 19-D  Pre-render animated PDP frames ──────────────────────────────────────
print('\n[4/5] Pre-rendering PDP animation...')
plot_frames=[]; perturbed_left=[]
for ai in range(len(ALPHA_GRID)):
    fig_a,axes_a=plt.subplots(1,NUM_ACTIONS,figsize=(3*NUM_ACTIONS,4),dpi=100)
    for a in range(NUM_ACTIONS):
        ax=axes_a[a]
        for si in range(XAI_N): ax.plot(ALPHA_GRID[:ai+1],ice_curves[si,:ai+1,a],alpha=0.3,lw=1.0,color=ACTION_COLORS[a])
        ax.plot(ALPHA_GRID[:ai+1],pdp_curves[:ai+1,a],lw=2.2,color='crimson')
        ax.set_xlim(ALPHA_GRID[0],ALPHA_GRID[-1])
        ax.set_ylim(ice_curves[:,:,a].min()-0.02,ice_curves[:,:,a].max()+0.02)
        ax.set_title(ACTION_NAMES[a],fontsize=8,fontweight='bold'); ax.set_xlabel('α',fontsize=7)
        if a==0: ax.set_ylabel('Pred. token',fontsize=7)
        ax.grid(alpha=0.25); ax.tick_params(labelsize=6)
    fig_a.suptitle(f'PDP (crimson) + ICE   α={ALPHA_GRID[ai]:.2f}',fontsize=10,fontweight='bold')
    plt.tight_layout(); plot_frames.append(fig_to_rgb(fig_a)); plt.close(fig_a)
    v_p=(xai_videos[0:1]*ALPHA_GRID[ai]).clamp(0,1)
    v_np=v_p[0,:,-1].cpu().numpy().transpose(1,2,0)
    perturbed_left.append(np.clip(v_np,0,1)); del v_p
for _ in range(HOLD_FRAMES):
    plot_frames.append(plot_frames[-1]); perturbed_left.append(perturbed_left[-1])

# ── 19-E  Stitch animation ────────────────────────────────────────────────────
print('\n[5/5] Stitching PDP animation...')
pf_h,pf_w=plot_frames[0].shape[:2]; pw=TARGET_SIZE+40; cw=pw+pf_w
ch=max(TARGET_SIZE+HBAR,pf_h+HBAR); HBAR=42; frames_rgb=[]
for fi in range(len(plot_frames)):
    canvas=np.zeros((ch,cw,3),dtype=np.uint8)
    alpha_val=ALPHA_GRID[min(fi,len(ALPHA_GRID)-1)]
    make_header(canvas,f"PDP/ICE sweep  α={alpha_val:.2f}  crimson=PDP  colour=ICE")
    v_uint8=cv2.resize((perturbed_left[fi]*255).astype(np.uint8),(pw,ch-HBAR))
    canvas[HBAR:,:pw]=v_uint8
    p_rsz=cv2.resize(plot_frames[fi],(pf_w,ch-HBAR))
    canvas[HBAR:,pw:]=p_rsz; frames_rgb.append(canvas)
save_video(frames_rgb,DIR19/'pdp_ice_animation',PDP_FPS)

block_summary(DIR19); print('\n✅ Block 19 done')


   VRAM free at PDP start: 5.48/6.09 GB
📊 BLOCK 19: PDP + ICE + Validation Improvement

[1/5] Evaluating trained model (n=20 batches)...
   Baseline (ln 256) : 5.5452
   Trained model     : 2.0382
   Δ                 : +3.5070  (+63.2%)
   Result            : ✅ IMPROVED
      💾 xai_outputs_rt1_lightweight_bow_convnext/block_19_pdp_ice/validation_improvement.png
[2/5] Computing PDP + ICE curves...
   α=0.54  (5/20)
   α=0.96  (10/20)
   α=1.38  (15/20)
   α=1.80  (20/20)
   ✅ done

[3/5] Static PDP + ICE plot...
      💾 xai_outputs_rt1_lightweight_bow_convnext/block_19_pdp_ice/pdp_ice_static.png

[4/5] Pre-rendering PDP animation...

[5/5] Stitching PDP animation...
      💾 xai_outputs_rt1_lightweight_bow_convnext/block_19_pdp_ice/pdp_ice_animation.gif  (0.8 MB)
      💾 xai_outputs_rt1_lightweight_bow_convnext/block_19_pdp_ice/pdp_ice_animation.mp4

   📁 block_19_pdp_ice/
      📄 pdp_ice_animation.gif  832 KB
      📄 pdp_ice_animation.mp4  477 KB
      📄 pdp_ice_static.png  874 KB
    

In [20]:
# ========================= BLOCK 20: NOVEL XAI FUSIONS =========================
DIR20=XAI_ROOT/'block_20_fusions'
H_img,W_img=xai_videos.shape[-2],xai_videos.shape[-1]

print('='*70); print('🔬 BLOCK 20: Novel XAI Fusions (ShaGCAM + LimeCAM)'); print('='*70)
torch.cuda.empty_cache(); gc.collect(); torch.cuda.empty_cache()
print(f'   Free VRAM: {torch.cuda.mem_get_info()[0]/1024**3:.2f} GB')

# ── 20-A  Static fusions ──────────────────────────────────────────────────────
print('\n[1/4] Static fusions on XAI_N samples...')
shagcam_results=[]; limecam_results=[]
for i in range(XAI_N):
    torch.cuda.empty_cache()
    frame_np=last_frame_np(xai_videos,i); frame_t=xai_videos[i:i+1,:,-1]
    with GradCAM(model=frame_model,target_layers=target_layer) as gcam:
        gray=gcam(input_tensor=frame_t,targets=cam_targets)[0]
    torch.cuda.empty_cache()
    gcam_map=resize_to(gray,H_img,W_img); gcam_norm=normalise(gcam_map)
    shap_norm=normalise(resize_to(shap_agg_vis[i],H_img,W_img))
    mask_2d=lime_results[i][1]; lime_norm=normalise(resize_to(mask_2d.astype(np.float32),H_img,W_img))
    sha_map=normalise(gcam_norm*shap_norm); sha_ov=show_cam_on_image(frame_np,sha_map,use_rgb=True)
    lc_map=normalise(np.sqrt(gcam_norm*(lime_norm+0.05))); lc_ov=show_cam_on_image(frame_np,lc_map,use_rgb=True)
    shagcam_results.append((frame_np,sha_map,sha_ov)); limecam_results.append((frame_np,lc_map,lc_ov))
    print(f'   sample {i+1}/{XAI_N} done')
torch.cuda.empty_cache()

fig,axes=plt.subplots(XAI_N,5,figsize=(25,5*XAI_N))
col_titles=['Original','GradCAM','|SHAP|','🔬 ShaGCAM\n(SHAP×GradCAM)','🔬 LimeCAM\n(√LIME×GradCAM)']
for j,t in enumerate(col_titles): axes[0,j].set_title(t,fontsize=10,fontweight='bold')
for i in range(XAI_N):
    frame_np=last_frame_np(xai_videos,i); _,_,gcam_ov=cam_results['GradCAM'][i]
    shap_vis=normalise(resize_to(shap_agg_vis[i],H_img,W_img))
    _,_,sha_ov=shagcam_results[i]; _,_,lc_ov=limecam_results[i]
    for col,(img,cmap) in enumerate(zip([frame_np,gcam_ov,shap_vis,sha_ov,lc_ov],[None,None,'hot',None,None])):
        axes[i,col].imshow(img,cmap=cmap); axes[i,col].axis('off')
    axes[i,0].set_ylabel(f'Sample {i+1}',fontsize=9)
plt.suptitle(f'Novel XAI Fusions — {MODEL_NAME}\nShaGCAM=SHAP×GradCAM | LimeCAM=√(LIME×GradCAM)',fontsize=13,fontweight='bold')
plt.tight_layout(); save_fig(fig,DIR20/'fusions_static.png')

# ── 20-B  Per-frame GradCAM ───────────────────────────────────────────────────
print(f'\n[2/4] Per-frame GradCAM ({VID_ACTUAL} frames)...')
torch.cuda.empty_cache(); fusion_gcam_maps=[]
with GradCAM(model=frame_model,target_layers=target_layer) as gcam:
    for f_idx in range(VID_ACTUAL):
        gray=gcam(input_tensor=vid_tensors[f_idx],targets=cam_targets)[0]
        fusion_gcam_maps.append(resize_to(gray,H_img,W_img))
        if (f_idx+1)%10==0: torch.cuda.empty_cache()
torch.cuda.empty_cache(); print('   ✅ GradCAM done')

# ── 20-C  Stitch 5-panel fusion video ─────────────────────────────────────────
print('\n[3/4] Stitching fusion video...')
H,W=H_img,W_img; HBAR=45; n_panels=5; cw,ch=W*n_panels,H+HBAR
labels=['Original','GradCAM','|SHAP|','🔬ShaGCAM','🔬LimeCAM']; frames_rgb=[]
for f_idx in range(VID_ACTUAL):
    frame_np=vid_numpys[f_idx]; gcam_norm=normalise(fusion_gcam_maps[f_idx])
    shap_n=(shap_norm_maps[f_idx] if f_idx<len(shap_norm_maps) else np.zeros((H,W),dtype=np.float32))
    shap_rs=normalise(resize_to(shap_n,H,W))
    lime_n=(normalise(resize_to(lime_masks[f_idx].astype(np.float32),H,W)) if f_idx<len(lime_masks) else np.zeros((H,W),dtype=np.float32))
    sha_map=normalise(gcam_norm*shap_rs); sha_ov=show_cam_on_image(frame_np,sha_map,use_rgb=True)
    lc_map=normalise(np.sqrt(gcam_norm*(lime_n+0.05))); lc_ov=show_cam_on_image(frame_np,lc_map,use_rgb=True)
    gcam_ov=show_cam_on_image(frame_np,gcam_norm,use_rgb=True)
    shap_ov=(np.clip(frame_np*0.45+plt.cm.hot(shap_rs)[:,:,:3].astype(np.float32)*0.55,0,1)*255).astype(np.uint8)
    panels=[(frame_np*255).astype(np.uint8),gcam_ov,shap_ov,sha_ov,lc_ov]
    canvas=np.zeros((ch,cw,3),dtype=np.uint8)
    canvas[:HBAR,:]=22
    cv2.putText(canvas,f"XAI Fusions | {f_idx+1}/{VID_ACTUAL} | '{VID_INSTRUCTION}'",(10,HBAR-12),cv2.FONT_HERSHEY_SIMPLEX,0.52,(220,220,220),1)
    for p_idx,(panel,lbl) in enumerate(zip(panels,labels)):
        x0=p_idx*W; canvas[HBAR:,x0:x0+W]=panel
        cv2.putText(canvas,lbl,(x0+4,HBAR+16),cv2.FONT_HERSHEY_SIMPLEX,0.42,(255,255,255),1)
    frames_rgb.append(canvas)
save_video(frames_rgb,DIR20/'fusions_video',VID_FPS)

# ── 20-D  Filmstrip ───────────────────────────────────────────────────────────
print('\n[4/4] Filmstrip...')
strip_idxs=list(range(0,VID_ACTUAL,10)); n_strip=len(strip_idxs)
fig,axes=plt.subplots(3,n_strip,figsize=(3*n_strip,9))
if n_strip==1: axes=axes[:,np.newaxis]
for col,f_idx in enumerate(strip_idxs):
    frame_np=vid_numpys[f_idx]; gcam_norm=normalise(fusion_gcam_maps[f_idx])
    shap_n=(shap_norm_maps[f_idx] if f_idx<len(shap_norm_maps) else np.zeros((H,W)))
    sha_map=normalise(gcam_norm*normalise(resize_to(shap_n,H,W))); sha_ov=show_cam_on_image(frame_np,sha_map,use_rgb=True)
    lime_n=(normalise(resize_to(lime_masks[f_idx].astype(np.float32),H,W)) if f_idx<len(lime_masks) else np.zeros((H,W)))
    lc_map=normalise(np.sqrt(gcam_norm*(lime_n+0.05))); lc_ov=show_cam_on_image(frame_np,lc_map,use_rgb=True)
    axes[0,col].imshow(frame_np); axes[0,col].set_title(f'F{f_idx+1}',fontsize=7); axes[0,col].axis('off')
    if col==0: axes[0,col].set_ylabel('Original',fontsize=9,fontweight='bold')
    axes[1,col].imshow(sha_ov); axes[1,col].axis('off')
    if col==0: axes[1,col].set_ylabel('ShaGCAM',fontsize=9,fontweight='bold')
    axes[2,col].imshow(lc_ov); axes[2,col].axis('off')
    if col==0: axes[2,col].set_ylabel('LimeCAM',fontsize=9,fontweight='bold')
plt.suptitle(f'Novel Fusions Filmstrip — "{VID_INSTRUCTION}"',fontsize=12,fontweight='bold')
plt.tight_layout(); save_fig(fig,DIR20/'fusions_filmstrip.png')

block_summary(DIR20); print('\n✅ Block 20 done')


🔬 BLOCK 20: Novel XAI Fusions (ShaGCAM + LimeCAM)
   Free VRAM: 5.11 GB

[1/4] Static fusions on XAI_N samples...
   sample 1/4 done
   sample 2/4 done
   sample 3/4 done
   sample 4/4 done
      💾 xai_outputs_rt1_lightweight_bow_convnext/block_20_fusions/fusions_static.png

[2/4] Per-frame GradCAM (100 frames)...
   ✅ GradCAM done

[3/4] Stitching fusion video...
      💾 xai_outputs_rt1_lightweight_bow_convnext/block_20_fusions/fusions_video.gif  (5.3 MB)
      💾 xai_outputs_rt1_lightweight_bow_convnext/block_20_fusions/fusions_video.mp4

[4/4] Filmstrip...
      💾 xai_outputs_rt1_lightweight_bow_convnext/block_20_fusions/fusions_filmstrip.png

   📁 block_20_fusions/
      📄 fusions_filmstrip.png  6.0 MB
      📄 fusions_static.png  2.3 MB
      📄 fusions_video.gif  5.3 MB
      📄 fusions_video.mp4  3.8 MB

✅ Block 20 done


In [21]:
# ========================= BLOCK 21: XAI MASTER DASHBOARD =========================
DIR21=XAI_ROOT/'block_21_dashboard'
print('='*70); print('🎛️  BLOCK 21: XAI Master Dashboard'); print('='*70)

sample_idx=0; frame_np_s=last_frame_np(xai_videos,sample_idx)
fig=plt.figure(figsize=(26,16)); fig.patch.set_facecolor('#0a0a14')

def dark_ax(ax,title=''):
    ax.set_facecolor('#13132a')
    for sp in ax.spines.values(): sp.set_edgecolor('#3a3a5c')
    ax.tick_params(colors='#aaa',labelsize=7)
    if title: ax.set_title(title,color='#e0e0ff',fontsize=9,fontweight='bold',pad=5)
    ax.axis('off')

gs=fig.add_gridspec(4,7,hspace=0.38,wspace=0.12,top=0.93,bottom=0.04,left=0.02,right=0.98)

# ── Row 0: Title ──────────────────────────────────────────────────────────────
ax_t=fig.add_subplot(gs[0,:]); ax_t.set_facecolor('#0a0a14'); ax_t.axis('off')
ax_t.text(0.5,0.62,f'🔬  XAI Dashboard — {MODEL_NAME}',ha='center',va='center',
          fontsize=18,color='white',fontweight='bold')
ax_t.text(0.5,0.22,f'Task: "{xai_insts[sample_idx]}"   |   Val loss: {current_val_loss:.4f}   |   Improvement vs random: {improvement:+.4f} ({pct:+.1f}%)',
          ha='center',va='center',fontsize=11,color='#aaddff')

# ── Row 1: core XAI methods ───────────────────────────────────────────────────
H_img2,W_img2=xai_videos.shape[-2],xai_videos.shape[-1]
row1_data=[('Original',frame_np_s,None),
           ('GradCAM',cam_results['GradCAM'][sample_idx][2],None),
           ('HiResCAM',cam_results['HiResCAM'][sample_idx][2],None),
           ('ShapleyCAM',cam_results['ShapleyCAM'][sample_idx][2],None),
           ('LIME top-5',lime_results[sample_idx][0],None),
           ('|SHAP|',normalise(resize_to(shap_agg_vis[sample_idx],H_img2,W_img2)),'hot')]
for col,(name,img,cmap) in enumerate(row1_data):
    ax=fig.add_subplot(gs[1,col]); dark_ax(ax,name); ax.imshow(img,cmap=cmap); ax.axis('off')
ax_val=fig.add_subplot(gs[1,6]); dark_ax(ax_val,'Val Loss'); ax_val.axis('on')
bars=ax_val.bar(['Random','Trained'],[baseline_val_loss,current_val_loss],color=['#cc4444','#44cc88'],width=0.5,edgecolor='#aaa')
for b in bars:
    ax_val.text(b.get_x()+b.get_width()/2,b.get_height()+0.01,f'{b.get_height():.3f}',ha='center',va='bottom',color='white',fontsize=9,fontweight='bold')
ax_val.set_facecolor('#13132a'); ax_val.tick_params(colors='#aaa',labelsize=8)
for sp in ax_val.spines.values(): sp.set_edgecolor('#3a3a5c')

# ── Row 2: fusions + evolution ────────────────────────────────────────────────
ax_sha=fig.add_subplot(gs[2,0:2]); dark_ax(ax_sha,'🔬 ShaGCAM  (SHAP × GradCAM)')
ax_sha.imshow(shagcam_results[sample_idx][2]); ax_sha.axis('off')
ax_lc=fig.add_subplot(gs[2,2:4]); dark_ax(ax_lc,'🔬 LimeCAM  (√LIME × GradCAM)')
ax_lc.imshow(limecam_results[sample_idx][2]); ax_lc.axis('off')
frame_np_c=frame_np_s.copy(); _,lmask,_=lime_results[sample_idx]
frame_np_c[lmask==1]=frame_np_c[lmask==1]*0.35+np.array([0.,0.85,0.2])*0.65
ax_lm=fig.add_subplot(gs[2,4]); dark_ax(ax_lm,'LIME Overlay'); ax_lm.imshow(np.clip(frame_np_c,0,1)); ax_lm.axis('off')
shap_n_s=normalise(resize_to(shap_agg_vis[sample_idx],H_img2,W_img2))
shap_b=np.clip(frame_np_s*0.45+plt.cm.hot(shap_n_s)[:,:,:3].astype(np.float32)*0.55,0,1)
ax_sb=fig.add_subplot(gs[2,5]); dark_ax(ax_sb,'SHAP Overlay'); ax_sb.imshow(shap_b); ax_sb.axis('off')
ax_ev=fig.add_subplot(gs[2,6]); dark_ax(ax_ev,'Attn Peak (100 frames)'); ax_ev.axis('on'); ax_ev.set_facecolor('#13132a')
for sp in ax_ev.spines.values(): sp.set_edgecolor('#3a3a5c')
for name in CAM_METHODS:
    ax_ev.plot(np.arange(1,VID_ACTUAL+1),attn_stats[name]['peak'],color=CAM_COLORS[name],lw=1.4,alpha=0.85,label=name)
ax_ev.set_xlabel('Frame',color='#aaa',fontsize=7); ax_ev.set_ylabel('Peak CAM',color='#aaa',fontsize=7)
ax_ev.tick_params(colors='#aaa',labelsize=6); ax_ev.legend(fontsize=6,labelcolor='white',facecolor='#13132a',edgecolor='#3a3a5c')
ax_ev.grid(alpha=0.15,color='#444')

# ── Row 3: PDP + LIME activity + mean SHAP ───────────────────────────────────
ax_pdp=fig.add_subplot(gs[3,0:3]); dark_ax(ax_pdp,'PDP — all actions vs brightness α'); ax_pdp.axis('on'); ax_pdp.set_facecolor('#13132a')
for sp in ax_pdp.spines.values(): sp.set_edgecolor('#3a3a5c')
for a in range(NUM_ACTIONS):
    ax_pdp.plot(ALPHA_GRID,pdp_curves[:,a],label=ACTION_NAMES[a],lw=2.0,color=plt.cm.tab10(a/NUM_ACTIONS))
ax_pdp.set_xlabel('Brightness α',color='#aaa',fontsize=8); ax_pdp.set_ylabel('Pred. token (norm)',color='#aaa',fontsize=8)
ax_pdp.tick_params(colors='#aaa',labelsize=7); ax_pdp.legend(fontsize=7,ncol=4,labelcolor='white',facecolor='#13132a',edgecolor='#3a3a5c')
ax_pdp.grid(alpha=0.15,color='#444')
active_arr_smooth=np.convolve(np.array(active_area),np.ones(10)/10,mode='same')
ax_la=fig.add_subplot(gs[3,3:5]); dark_ax(ax_la,'LIME Active Area (100 frames)'); ax_la.axis('on'); ax_la.set_facecolor('#13132a')
for sp in ax_la.spines.values(): sp.set_edgecolor('#3a3a5c')
ax_la.fill_between(np.arange(1,VID_ACTUAL+1),np.array(active_area),alpha=0.2,color='limegreen')
ax_la.plot(np.arange(1,VID_ACTUAL+1),active_arr_smooth,color='limegreen',lw=2.0)
ax_la.set_xlabel('Frame',color='#aaa',fontsize=8); ax_la.set_ylabel('Fraction highlighted',color='#aaa',fontsize=8)
ax_la.tick_params(colors='#aaa',labelsize=7); ax_la.grid(alpha=0.15,color='#444')
ax_ms=fig.add_subplot(gs[3,5:]); dark_ax(ax_ms,'Mean |SHAP| — 100 frames'); ax_ms.axis('on')
ax_ms.imshow(shap_norm_maps.mean(axis=0),cmap='hot',vmin=0,vmax=1); ax_ms.axis('off')

dash_path=str(DIR21/'xai_dashboard.png')
plt.savefig(dash_path,dpi=180,bbox_inches='tight',facecolor=fig.get_facecolor())
plt.show(); plt.close(fig); print(f'   💾 {dash_path}')

# ── Final tree ────────────────────────────────────────────────────────────────
print(f'\n{"="*70}')
print(f'🎉 ALL XAI BLOCKS COMPLETE — {MODEL_NAME}')
print(f'{"="*70}')
total_kb=0
for sub in sorted(XAI_ROOT.iterdir()):
    if not sub.is_dir(): continue
    sub_kb=sum(f.stat().st_size for f in sub.iterdir())/1024; total_kb+=sub_kb
    print(f'\n   📁 {sub.name}/  ({sub_kb/1024:.1f} MB)')
    for f in sorted(sub.iterdir()):
        kb=f.stat().st_size/1024
        print(f'      📄 {f.name:<45} {"%.1f MB" % (kb/1024) if kb>1024 else "%.0f KB" % kb}')
print(f'\n   Total: {total_kb/1024:.1f} MB  |  Root: {XAI_ROOT.resolve()}')
print(f'{"="*70}')
print("✅ Block 21 done")


🎛️  BLOCK 21: XAI Master Dashboard
   💾 xai_outputs_rt1_lightweight_bow_convnext/block_21_dashboard/xai_dashboard.png

🎉 ALL XAI BLOCKS COMPLETE — rt1_lightweight_bow_convnext

   📁 block_16_gradcam/  (16.2 MB)
      📄 gradcam_evolution.png                         182 KB
      📄 gradcam_filmstrip.png                         6.2 MB
      📄 gradcam_sidebyside.gif                        3.5 MB
      📄 gradcam_sidebyside.mp4                        1.8 MB
      📄 gradcam_static.png                            4.3 MB
      📄 gradcam_trajectory.png                        193 KB

   📁 block_17_lime/  (8.1 MB)
      📄 lime_activity.png                             684 KB
      📄 lime_filmstrip.png                            3.3 MB
      📄 lime_static.png                               890 KB
      📄 lime_video.gif                                2.0 MB
      📄 lime_video.mp4                                1.3 MB

   📁 block_18_shap/  (16.1 MB)
      📄 shap_filmstrip.png                            6